# Crumple Video - Blender XPBD Paper Animation

Генерация видеоанонса с эффектом раскрывающейся бумаги.

**Особенности:**
- Полная Blender XPBD физика на CPU
- Обложка с датами и типографикой
- Аудиотрек The xx - Intro (старт с 1:10)

**Версия v2.0 - с полными XPBD скриптами**

In [ ]:
NOTEBOOK_VERSION = 'v2.1-blender36'
print(f'🚀 Engine {NOTEBOOK_VERSION}')

# ==========================================
# 1. УСТАНОВКА ЗАВИСИМОСТЕЙ (Blender 3.6)
# ==========================================
import warnings
warnings.filterwarnings("ignore")

print("⏳ Installing dependencies...", flush=True)

# Install FFmpeg from apt
!apt-get update -qq && apt-get install -y -qq ffmpeg libxrender1 libxi6 libxkbcommon-x11-0 > /dev/null 2>&1

# Install Blender 3.6 LTS from tarball (compatible with XPBD script)
import os
if not os.path.exists('/opt/blender-3.6'):
    print('📦 Downloading Blender 3.6 LTS...', flush=True)
    !wget -q https://download.blender.org/release/Blender3.6/blender-3.6.16-linux-x64.tar.xz -O /tmp/blender.tar.xz
    !mkdir -p /opt && tar -xf /tmp/blender.tar.xz -C /opt
    !mv /opt/blender-3.6.* /opt/blender-3.6
    !ln -sf /opt/blender-3.6/blender /usr/local/bin/blender
    !rm /tmp/blender.tar.xz
else:
    print('✅ Blender 3.6 already installed')

# Verify Blender version
!blender --version | head -2

!pip install opencv-python numpy pillow requests telethon cryptography > /dev/null

import os
import sys
import json
import glob
import shutil
import subprocess
from pathlib import Path

import numpy as np
import cv2
from PIL import Image, ImageFont, ImageDraw

def log(msg):
    print(msg, flush=True)

log("✅ Dependencies ready")


In [ ]:
# ==========================================
# 2. КОНФИГУРАЦИЯ
# ==========================================
KAGGLE_INPUT_ROOT = "/kaggle/input"
WORKING_DIR = Path("/kaggle/working")

session_dirs = []
for pattern in ["video-session-*", "video-afisha-session-*"]:
    session_dirs.extend(glob.glob(os.path.join(KAGGLE_INPUT_ROOT, pattern)))
session_dirs = sorted(set(session_dirs), reverse=True)
crumple_test_dirs = sorted(glob.glob(os.path.join(KAGGLE_INPUT_ROOT, "crumple-video-test*")), reverse=True)

if session_dirs:
    SOURCE_FOLDER = Path(session_dirs[0])
    log(f"✅ DETECTED BOT SESSION: {SOURCE_FOLDER}")
elif crumple_test_dirs:
    SOURCE_FOLDER = Path(crumple_test_dirs[0])
    log(f"✅ DETECTED TEST DATASET: {SOURCE_FOLDER}")
elif os.path.exists(os.path.join(KAGGLE_INPUT_ROOT, "afisha-dataset-2")):
    SOURCE_FOLDER = Path(os.path.join(KAGGLE_INPUT_ROOT, "afisha-dataset-2"))
    log(f"⚠️ Using legacy dataset: {SOURCE_FOLDER}")
else:
    all_dirs = [d for d in glob.glob(os.path.join(KAGGLE_INPUT_ROOT, "*")) if os.path.isdir(d)]
    SOURCE_FOLDER = Path(all_dirs[0]) if all_dirs else Path("/kaggle/input/afisha-dataset-2")
    log(f"⚠️ Fallback: {SOURCE_FOLDER}")

# Robust payload detection (prefer session/test datasets)
payload_path = SOURCE_FOLDER / "payload.json"
if payload_path.exists():
    log(f"✅ Found payload.json at {payload_path}")
else:
    print("DEBUG: Searching for payload.json in", KAGGLE_INPUT_ROOT)
    payloads = list(Path(KAGGLE_INPUT_ROOT).rglob("payload.json"))
    if payloads:
        def _payload_score(p: Path) -> tuple[int, float, str]:
            name = p.parent.name.lower()
            if name.startswith("video-session-") or name.startswith("video-afisha-session-"):
                priority = 0
            elif name.startswith("crumple-video-test"):
                priority = 1
            elif "afisha-dataset" in name:
                priority = 9
            else:
                priority = 5
            try:
                mtime = p.stat().st_mtime
            except Exception:
                mtime = 0.0
            return (priority, -mtime, p.as_posix())
        payloads.sort(key=_payload_score)
        SOURCE_FOLDER = payloads[0].parent
        log(f"✅ Found payload.json at {payloads[0]}, setting SOURCE_FOLDER to {SOURCE_FOLDER}")
    else:
        log("⚠️ payload.json not found via rglob, listing input dir:")
        for root, dirs, files in os.walk(KAGGLE_INPUT_ROOT):
            for f in files:
                print(os.path.join(root, f))

# Параметры видео
W, H = 1080, 1572  # CrumpleVideo canonical aspect
FPS = 24

# Аудио
AUDIO_START_SEC = 77  # 1:17 от начала трека
AUDIO_FILENAME = "The_xx_-_Intro.mp3"

# Blender params - больше кадров для плавности
FRAME_START = 1
FRAME_END = 64
BLENDER_SAMPLES = 48  # Более высокое качество
UNFOLD_DELAY = 0.15
UNFOLD_GAMMA = 0.9
SIMILARITY_THRESHOLD = 0.998
BLENDER_RENDER_TIMEOUT_SECONDS = 7200  # 120 min per poster for CPU fallback runs

WORKING_DIR.mkdir(parents=True, exist_ok=True)
log("✅ Config ready")


In [ ]:

import sys
import os
import shutil
import re
from pathlib import Path
from datetime import date, timedelta

# Fix: Copy assets from read-only input to working dir to allow proper importing
possible_paths = [
    Path("/kaggle/input/video-announce-assets"),
    Path("/kaggle/input/video-announce-assets/video-announce-assets"),
]

dataset_dir = None
for p in possible_paths:
    if p.exists():
        dataset_dir = p
        break

working_dir = Path("/kaggle/working")
if dataset_dir:
    print(f"✅ Found assets dataset at {dataset_dir}")
    
    # Copy pattern_preview.py
    pp_src = dataset_dir / "pattern_preview.py"
    if pp_src.exists():
        shutil.copy2(pp_src, working_dir / "pattern_preview.py")
        print("✅ Copied pattern_preview.py to working dir")
    else:
        print(f"⚠️ pattern_preview.py not found in {dataset_dir}")

    # Copy assets folder
    assets_src = dataset_dir / "assets"
    assets_dst = working_dir / "assets"
    
    if assets_src.exists():
        if assets_dst.exists():
            shutil.rmtree(assets_dst)
        shutil.copytree(assets_src, assets_dst)
        print("✅ Copied assets folder to working dir")
    else:
        # Fallback for loose files
        os.makedirs(assets_dst, exist_ok=True)
        copied_any = False
        for ext in ["*.ttf", "*.otf"]:
            for f in dataset_dir.glob(ext):
                shutil.copy2(f, assets_dst)
                copied_any = True
        if copied_any:
            print("✅ Copied loose font files to assets/ folder")
        else:
            print("⚠️ assets folder/files not found in dataset")
else:
    print("⚠️ Dataset video-announce-assets NOT FOUND")

pattern_preview = None
try:
    import pattern_preview as _pattern_preview
    pattern_preview = _pattern_preview
    print(f"✅ Imported pattern_preview from {pattern_preview.__file__}")
except ImportError as e:
    print(f"⚠️ Could not import pattern_preview: {e}")


def _normalize_pattern(pattern: str) -> str:
    if pattern_preview and hasattr(pattern_preview, 'ALL_PATTERNS') and pattern in pattern_preview.ALL_PATTERNS:
        return pattern
    return getattr(pattern_preview, 'PATTERN_STICKER', 'STICKER')


_ASSETS_DIR = Path("/kaggle/working/assets")
if not _ASSETS_DIR.exists():
    _ASSETS_DIR = Path("/kaggle/input/video-announce-assets/assets")

_FONT_BENZIN = _ASSETS_DIR / "Benzin-Bold.ttf"
_FONT_BEBAS = _ASSETS_DIR / "BebasNeue-Bold.ttf"
_FONT_OSWALD = _ASSETS_DIR / "Oswald-VariableFont_wght.ttf"
_FONT_DRUK = _ASSETS_DIR / "DrukCyr-Bold.ttf"

_MONTHS_RU = [
    "января",
    "февраля",
    "марта",
    "апреля",
    "мая",
    "июня",
    "июля",
    "августа",
    "сентября",
    "октября",
    "ноября",
    "декабря",
]

_WEEKDAYS_RU = [
    "ПОНЕДЕЛЬНИК",
    "ВТОРНИК",
    "СРЕДА",
    "ЧЕТВЕРГ",
    "ПЯТНИЦА",
    "СУББОТА",
    "ВОСКРЕСЕНЬЕ",
]


def _parse_iso_date(value):
    if not value:
        return None
    try:
        return date.fromisoformat(str(value))
    except Exception:
        return None


def _parse_date_text(raw_text: str | None, month_hint: str | None = None):
    if not raw_text:
        return (None, None)
    text = str(raw_text).strip().lower()
    month = None
    for idx, name in enumerate(_MONTHS_RU, start=1):
        if name in text:
            month = idx
            break
    if not month and month_hint:
        hint = str(month_hint).strip().lower()
        for idx, name in enumerate(_MONTHS_RU, start=1):
            if name in hint:
                month = idx
                break
    if not month:
        return (None, None)

    year = date.today().year
    match_range = re.search(r"(\d{1,2})\s*[-–—]\s*(\d{1,2})", text)
    if match_range:
        start_day = int(match_range.group(1))
        end_day = int(match_range.group(2))
        try:
            return (date(year, month, start_day), date(year, month, end_day))
        except Exception:
            return (None, None)
    match_single = re.search(r"(\d{1,2})", text)
    if match_single:
        day = int(match_single.group(1))
        try:
            return (date(year, month, day), date(year, month, day))
        except Exception:
            return (None, None)
    return (None, None)


def _intro_layout_meta(intro: dict, cover: dict):
    date_start = _parse_iso_date(intro.get("date_start") or cover.get("date_start"))
    date_end = _parse_iso_date(intro.get("date_end") or cover.get("date_end"))

    if not date_start and not date_end:
        date_text = intro.get("date") or cover.get("date") or cover.get("date_range")
        month_hint = intro.get("month") or cover.get("month")
        date_start, date_end = _parse_date_text(date_text, month_hint=month_hint)

    if not date_start and not date_end:
        return None
    if not date_start:
        date_start = date_end
    if not date_end:
        date_end = date_start
    if not date_start or not date_end:
        return None

    if date_start == date_end:
        return {
            "layout": "day",
            "date_text": f"{date_start.day}",
            "month_text": _MONTHS_RU[date_start.month - 1],
            "title_text": _WEEKDAYS_RU[date_start.weekday()],
        }

    if date_start.month != date_end.month:
        return {
            "layout": "cross_month",
            "start_day": f"{date_start.day}",
            "start_month": f"{_MONTHS_RU[date_start.month - 1].upper()} —",
            "end_day": f"{date_end.day}",
            "end_month": _MONTHS_RU[date_end.month - 1].upper(),
        }

    weekend_only = True
    cur = date_start
    while cur <= date_end:
        if cur.weekday() < 5:
            weekend_only = False
            break
        cur += timedelta(days=1)
    if weekend_only:
        title_text = "ВЫХОДНЫЕ"
    else:
        title_text = f"{_WEEKDAYS_RU[date_start.weekday()]} — {_WEEKDAYS_RU[date_end.weekday()]}"

    return {
        "layout": "range",
        "date_text": f"{date_start.day}-{date_end.day}",
        "month_text": _MONTHS_RU[date_start.month - 1],
        "title_text": title_text,
    }

def _draw_text_right(draw, text, font, box, fill):
    x, y, w, _ = box
    text_w = draw.textlength(text, font=font)
    draw.text((x + w - text_w, y), text, font=font, fill=fill)


def _draw_multiline_right(draw, lines, font, box, line_height, fill):
    x, y, w, _ = box
    for idx, line in enumerate(lines):
        text_w = draw.textlength(line, font=font)
        draw.text((x + w - text_w, y + idx * line_height), line, font=font, fill=fill)


def _draw_rotated_text(base, text, font, fill, position, angle):
    bbox = font.getbbox(text)
    text_w = bbox[2] - bbox[0]
    text_h = bbox[3] - bbox[1]
    pad = max(6, int(min(text_w, text_h) * 0.04))
    layer = Image.new("RGBA", (text_w + pad * 2, text_h + pad * 2), (0, 0, 0, 0))
    draw = ImageDraw.Draw(layer)
    draw.text((pad - bbox[0], pad - bbox[1]), text, font=font, fill=fill)
    rotated = layer.rotate(angle, expand=True, resample=Image.BICUBIC)
    base.paste(rotated, (position[0] - pad, position[1] - pad), rotated)


def _render_css_intro(meta, cities, output_path: Path) -> Path:
    width, height = 1080, 1572
    bg = (241, 228, 75)
    fg = (16, 14, 14)

    img = Image.new("RGB", (width, height), bg)
    draw = ImageDraw.Draw(img)

    font_date = ImageFont.truetype(str(_FONT_BENZIN), 224)
    font_month = ImageFont.truetype(str(_FONT_BEBAS), 200)
    font_city = ImageFont.truetype(str(_FONT_OSWALD), 60)
    font_title_weekend = ImageFont.truetype(str(_FONT_DRUK), 220)
    font_title_day = ImageFont.truetype(str(_FONT_DRUK), 180)

    cities_list = []
    if isinstance(cities, str):
        cities_list = [c.strip() for c in cities.split(",") if c.strip()]
    else:
        cities_list = [c.strip() for c in (cities or []) if c]
    cities_list = [c.upper() for c in cities_list][:3]

    def _split_weekday_range(title_text: str):
        raw = str(title_text or "").strip().upper()
        if not raw:
            return None
        # Expected: "ПОНЕДЕЛЬНИК — СРЕДА" (we keep order start->end).
        parts = re.split(r"\s*[\-–—]\s*", raw, maxsplit=1)
        if len(parts) != 2:
            return None
        a, b = (parts[0].strip(), parts[1].strip())
        if not a or not b:
            return None
        # Put the dash on the line with the shorter weekday name.
        if len(a) <= len(b):
            return [f"{a} —", b]
        return [a, f"— {b}"]

    def _measure(draw, text: str, font):
        bbox = draw.textbbox((0, 0), text, font=font)
        return (bbox[2] - bbox[0], bbox[3] - bbox[1])

    def _draw_multiline_left(draw, x: int, y: int, lines: list[str], font, gap: int, fill):
        yy = y
        for i, line in enumerate(lines):
            bbox = draw.textbbox((0, 0), line, font=font)
            draw.text((x, yy), line, font=font, fill=fill)
            if i != len(lines) - 1:
                next_bbox = draw.textbbox((0, 0), lines[i + 1], font=font)
                yy = (yy + bbox[3]) + gap - next_bbox[1]

    def _draw_multiline_right_var(
        draw,
        x: int,
        y: int,
        lines: list[str],
        font,
        gap: int,
        fill,
        box_w: int,
    ):
        yy = y
        for i, line in enumerate(lines):
            bbox = draw.textbbox((0, 0), line, font=font)
            text_w = bbox[2] - bbox[0]
            draw.text((x + box_w - text_w, yy), line, font=font, fill=fill)
            if i != len(lines) - 1:
                next_bbox = draw.textbbox((0, 0), lines[i + 1], font=font)
                yy = (yy + bbox[3]) + gap - next_bbox[1]

    def _layout_weekday_title(draw, title_lines: list[str], *, max_w: int, cities_y0: int | None):
        # Keep the weekday range title large and avoid collisions with the cities block.
        # Strategy: prefer moving the title up; if still not enough, push cities down (within canvas).
        title_x = 82
        preferred_title_y = 779
        # Date block ends at 270+308=578; keep a conservative gap.
        min_title_y = 620
        gap = 12
        min_gap_to_cities = 40
        bottom_margin = 40

        cities_line_h = 89

        def _block_bottom(y0: int, bboxes: list[tuple[int, int, int, int]], gap_px: int) -> int:
            yy = y0
            for i, bbox in enumerate(bboxes):
                bottom = yy + bbox[3]
                if i == len(bboxes) - 1:
                    return bottom
                next_bbox = bboxes[i + 1]
                yy = bottom + gap_px - next_bbox[1]
            return y0

        # Cities pixel metrics (PIL textbbox uses a non-zero top offset for many fonts).
        cities_top_off = 0
        base_cities_y0 = None
        max_cities_y0 = None
        cities_top_px = None
        if cities_y0 is not None and cities_list:
            city_bboxes = [draw.textbbox((0, 0), c, font=font_city) for c in cities_list]
            cities_top_off = min(b[1] for b in city_bboxes)
            cities_bottom_off = max(b[3] for b in city_bboxes)
            cities_pixel_h = cities_bottom_off + cities_line_h * (len(cities_list) - 1)
            max_cities_y0 = height - bottom_margin - cities_pixel_h
            base_cities_y0 = min(int(cities_y0), int(max_cities_y0))
            cities_top_px = base_cities_y0 + cities_top_off

        best = None
        for size in range(220, 119, -2):
            font = ImageFont.truetype(str(_FONT_DRUK), int(size))
            title_bboxes = [draw.textbbox((0, 0), line, font=font) for line in title_lines]
            widths = [(b[2] - b[0]) for b in title_bboxes]
            if max(widths) > max_w:
                continue

            # No cities: keep within bottom margin.
            if cities_y0 is None:
                for y in range(preferred_title_y, min_title_y - 1, -4):
                    bottom_px = _block_bottom(y, title_bboxes, gap)
                    if bottom_px <= height - bottom_margin:
                        best = (title_x, y, font, gap, None, max_w)
                        break
                if best is not None:
                    break
                continue

            # With cities: 1) try moving the title up while keeping cities fixed.
            for y in range(preferred_title_y, min_title_y - 1, -4):
                bottom_px = _block_bottom(y, title_bboxes, gap)
                if bottom_px + min_gap_to_cities <= cities_top_px:
                    best = (title_x, y, font, gap, base_cities_y0, max_w)
                    break
            if best is not None:
                break

            # 2) push cities down just enough (but not beyond max_cities_y0).
            for y in range(preferred_title_y, min_title_y - 1, -4):
                bottom_px = _block_bottom(y, title_bboxes, gap)
                needed_cities_y0 = bottom_px + min_gap_to_cities - cities_top_off
                needed_cities_y0 = int(max(needed_cities_y0, base_cities_y0))
                if needed_cities_y0 <= max_cities_y0:
                    best = (title_x, y, font, gap, needed_cities_y0, max_w)
                    break
            if best is not None:
                break

        if best is None:
            # Hard fallback: smaller font + conservative placement.
            font = ImageFont.truetype(str(_FONT_DRUK), 120)
            gap_fallback = 10
            title_bboxes = [draw.textbbox((0, 0), line, font=font) for line in title_lines]

            if cities_y0 is None:
                chosen_y = min_title_y
                for y in range(preferred_title_y, min_title_y - 1, -4):
                    if _block_bottom(y, title_bboxes, gap_fallback) <= height - bottom_margin:
                        chosen_y = y
                        break
                best = (82, chosen_y, font, gap_fallback, None, max_w)
            else:
                chosen_y = min_title_y
                chosen_cities_y0 = base_cities_y0
                for y in range(preferred_title_y, min_title_y - 1, -4):
                    bottom_px = _block_bottom(y, title_bboxes, gap_fallback)
                    needed_cities_y0 = int(max(bottom_px + min_gap_to_cities - cities_top_off, base_cities_y0))
                    if needed_cities_y0 <= max_cities_y0:
                        chosen_y = y
                        chosen_cities_y0 = needed_cities_y0
                        break
                best = (82, chosen_y, font, gap_fallback, chosen_cities_y0, max_w)

        return best

    if meta["layout"] in ("weekend", "range"):
        _draw_text_right(draw, meta["date_text"], font_date, (55, 270, 945, 308), fg)
        _draw_rotated_text(img, meta["month_text"], font_month, fg, (850, 541), -90)
        title_text = meta.get("title_text") or ""
        cities_y0 = 1058 if cities_list else None
        weekday_lines = _split_weekday_range(title_text)
        if weekday_lines:
            x, y, font, gap, cities_y0, box_w = _layout_weekday_title(draw, weekday_lines, max_w=710, cities_y0=cities_y0)
            _draw_multiline_right_var(draw, x, y, weekday_lines, font, gap, fg, box_w)
        else:
            draw.text((82, 779), str(title_text), font=font_title_weekend, fill=fg)
        if cities_list:
            _draw_multiline_right(draw, cities_list, font_city, (435, int(cities_y0 or 1058), 357, 267), 89, fg)
    elif meta["layout"] == "day":
        _draw_text_right(draw, meta["date_text"], font_date, (676, 270, 324, 308), fg)
        _draw_rotated_text(img, meta["month_text"], font_month, fg, (850, 541), -90)
        _draw_text_right(draw, meta["title_text"], font_title_day, (73, 827, 724, 228), fg)
        if cities_list:
            _draw_multiline_right(draw, cities_list, font_city, (435, 1058, 357, 267), 89, fg)
    elif meta["layout"] == "cross_month":
        _draw_text_right(draw, meta["start_day"], font_title_day, (157, 637, 107, 228), fg)
        draw.text((317, 637), meta["start_month"], font=font_title_day, fill=fg)
        _draw_text_right(draw, meta["end_day"], font_title_day, (220, 827, 44, 228), fg)
        draw.text((317, 827), meta["end_month"], font=font_title_day, fill=fg)
        if cities_list:
            _draw_multiline_right(draw, cities_list, font_city, (435, 1058, 357, 267), 89, fg)

    out_path = Path(output_path)
    img.save(out_path, format="PNG")
    print(f"✅ Saved intro preview: {out_path}")
    return out_path


def generate_intro_image(
    pattern: str,
    intro_text: str,
    cities: list[str] | str | None,
    output_path: Path,
    intro: dict | None = None,
    cover: dict | None = None,
) -> Path:
    meta = _intro_layout_meta(intro or {}, cover or {})
    if meta:
        return _render_css_intro(meta, cities, output_path)

    if pattern_preview is None:
        raise RuntimeError("pattern_preview not available")

    normalized = _normalize_pattern(str(pattern or ""))
    if isinstance(cities, str):
        cities_str = cities
    else:
        cities_str = ", ".join([c for c in (cities or []) if c]) or None
    intro_text = intro_text or "АФИША"

    image_bytes = pattern_preview.generate_intro_preview(normalized, intro_text, cities_str)

    out_path = Path(output_path)
    out_path.write_bytes(image_bytes)
    print(f"✅ Saved intro preview: {out_path}")
    return out_path


In [ ]:
# ==========================================
# 4. BLENDER XPBD SCRIPT (встроенный)
# ==========================================
import base64

# Полный blender_xpbd_paper.py (780 строк) закодирован в base64
BLENDER_SCRIPT_B64 = """..."""

# Декодируем и сохраняем скрипт
BLENDER_SCRIPT_PATH = WORKING_DIR / "blender_xpbd_paper.py"
BLENDER_SCRIPT_PATH.write_bytes(base64.b64decode(BLENDER_SCRIPT_B64))
log(f"✅ Embedded Blender XPBD script written: {BLENDER_SCRIPT_PATH} ({BLENDER_SCRIPT_PATH.stat().st_size} bytes)")

def run_blender_render(
    poster_path: Path,
    out_dir: Path,
    samples: int,
    pct: int,
    frame_start: int = FRAME_START,
    frame_end: int = FRAME_END,
    extra_args: dict | None = None,
) -> bool:
    """Run Blender in background to render frames."""
    if not BLENDER_SCRIPT_PATH.exists():
        log(f"❌ Blender script not found: {BLENDER_SCRIPT_PATH}")
        return False
    
    out_dir.mkdir(parents=True, exist_ok=True)
    
    cmd = [
        "blender", "-b", "--factory-startup", "-noaudio",
        "-P", str(BLENDER_SCRIPT_PATH),
        "--",
        "--poster", str(poster_path),
        "--out_dir", str(out_dir),
        "--frame_start", str(frame_start),
        "--frame_end", str(frame_end),
        "--unfold_delay", str(UNFOLD_DELAY),
        "--unfold_gamma", str(UNFOLD_GAMMA),
        "--samples", str(samples),
        "--pct", str(pct),
    ]
    if extra_args:
        for key, value in extra_args.items():
            if value is None:
                continue
            cmd.extend([f"--{key}", str(value)])
    log(f"🎬 Rendering: {poster_path.name}")
    timeout_minutes = max(1, int(BLENDER_RENDER_TIMEOUT_SECONDS // 60))
    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=BLENDER_RENDER_TIMEOUT_SECONDS,
        )
        if result.returncode != 0:
            log(f"❌ Blender error (exit {result.returncode}):")
            log(result.stderr[:1000] if result.stderr else "No stderr")
            return False
    except subprocess.TimeoutExpired as exc:
        partial_frames = len(list(out_dir.glob("frame_*.png")))
        log(
            f"❌ Blender timeout ({timeout_minutes} min, partial_frames={partial_frames})"
        )
        stderr = (exc.stderr or "")[-1000:] if getattr(exc, "stderr", None) else ""
        if stderr:
            log(stderr)
        return False
    except Exception as e:
        log(f"❌ Blender exception: {e}")
        return False
    
    # Проверяем что кадры созданы
    frames = list(out_dir.glob("frame_*.png"))
    if not frames:
        log("❌ No frames generated")
        return False
    
    # Проверяем размер первого кадра
    first_frame = sorted(frames)[0]
    frame_size = first_frame.stat().st_size
    if frame_size < 10000:
        log(f"⚠️ Frame too small ({frame_size} bytes)")
    
    log(f"✅ Done: {len(frames)} frames, first={frame_size/1024:.1f}KB")
    return True

log("✅ Blender runner ready")


In [ ]:
# ==========================================
# 5. SHOWREEL ASSEMBLY (из make_showreel_audio.py)
# ==========================================
from dataclasses import dataclass
import math

def blend(a, b, t: float):
    """Blend two frames with weight t (0.0 = a, 1.0 = b)"""
    t = float(max(0.0, min(1.0, t)))
    return cv2.addWeighted(a, 1.0 - t, b, t, 0.0)

def ease_linear(x: float) -> float:
    return x

def ease_in_quad(x: float) -> float:
    return x * x

def ease_out_quad(x: float) -> float:
    return 1.0 - (1.0 - x) * (1.0 - x)

def ease_in_out_cubic(x: float) -> float:
    if x < 0.5:
        return 4.0 * x * x * x
    return 1.0 - pow(-2.0 * x + 2.0, 3.0) / 2.0

def ease_out_expo(x: float) -> float:
    return 1.0 if x >= 1.0 else 1.0 - pow(2, -10 * x)

def ease_in_out_back(x: float) -> float:
    c1 = 1.70158
    c2 = c1 * 1.525
    if x < 0.5:
        return (pow(2 * x, 2) * ((c2 + 1) * 2 * x - c2)) / 2
    return (pow(2 * x - 2, 2) * ((c2 + 1) * (x * 2 - 2) + c2) + 2) / 2

EASING = {
    "linear": ease_linear,
    "ease_in_quad": ease_in_quad,
    "ease_out_quad": ease_out_quad,
    "ease_in_out_cubic": ease_in_out_cubic,
    "ease_out_expo": ease_out_expo,
    "ease_in_out_back": ease_in_out_back,
}

def render_motion(base: list, length: int, easing_name: str, reverse: bool = False, frame_interp: bool = True) -> list:
    """Render motion with proper easing and frame interpolation."""
    if length <= 0 or not base:
        return []
    if len(base) == 1:
        return [base[0]] * int(length)
    
    ease = EASING.get(easing_name, ease_in_out_cubic)
    out = []
    n = max(1, int(length))
    
    for i in range(n):
        t = 0.0 if n == 1 else (i / (n - 1))
        s = float(max(0.0, min(1.0, ease(t)))) * (len(base) - 1)
        if reverse:
            s = (len(base) - 1) - s
        
        if not frame_interp:
            idx = int(round(s))
            out.append(base[idx])
        else:
            i0 = int(math.floor(s))
            i1 = min(len(base) - 1, i0 + 1)
            alpha = float(s - i0)
            out.append(blend(base[i0], base[i1], alpha))
    
    return out

# Timing parameters from original make_showreel_audio.py
UNFOLD_LEN = 24
FOLD_LEN = 24
HOLD_FLAT = 2
HOLD_BALL = 2
FRAME_INTERP = True

@dataclass
class SegmentPlan:
    stem: str
    unfold_len: int = UNFOLD_LEN
    fold_len: int = FOLD_LEN
    hold_flat: int = HOLD_FLAT
    hold_ball: int = HOLD_BALL
    easing_unfold: str = "ease_in_out_back"
    easing_fold: str = "ease_out_quad"

def composite_rgba_on_bg(rgba_frame, bg_color=(20, 20, 25)):
    """Composite RGBA frame onto dark gradient background."""
    h, w = rgba_frame.shape[:2]
    bg = np.zeros((h, w, 3), dtype=np.uint8)
    for y in range(h):
        factor = 0.8 + 0.2 * (y / h)
        bg[y, :] = [int(c * factor) for c in bg_color]
    
    if rgba_frame.shape[2] == 4:
        alpha = rgba_frame[:, :, 3:4].astype(np.float32) / 255.0
        rgb = rgba_frame[:, :, :3].astype(np.float32)
        result = (rgb * alpha + bg.astype(np.float32) * (1 - alpha))
        return result.astype(np.uint8)
    else:
        return rgba_frame[:, :, :3]

def _prep_similarity(frame):
    small = cv2.resize(frame, (96, 96), interpolation=cv2.INTER_AREA)
    return cv2.cvtColor(small, cv2.COLOR_RGB2GRAY)

def frame_similarity(a, b) -> float:
    if a is None or b is None:
        return 0.0
    ga = _prep_similarity(a)
    gb = _prep_similarity(b)
    if ga.shape != gb.shape:
        return 0.0
    diff = cv2.absdiff(ga, gb)
    mean = float(diff.mean()) / 255.0
    return 1.0 - mean

def load_segment_frames(
    frames_dir: Path,
    frame_start: int,
    frame_end: int,
    similarity_threshold: float = SIMILARITY_THRESHOLD,
) -> list:
    """Load and composite Blender frames for a segment."""
    frames = []
    last = None
    dropped = 0
    for fn in range(frame_start, frame_end + 1):
        frame_path = frames_dir / f"frame_{fn:04d}.png"
        if not frame_path.exists():
            continue
        img = cv2.imread(str(frame_path), cv2.IMREAD_UNCHANGED)
        if img is None:
            continue
        composited = composite_rgba_on_bg(img)
        rgb = cv2.cvtColor(composited, cv2.COLOR_BGR2RGB)
        if similarity_threshold is not None and last is not None:
            if frame_similarity(last, rgb) >= similarity_threshold:
                dropped += 1
                continue
        frames.append(rgb)
        last = rgb
    if dropped:
        log(f"⚠️ Dropped {dropped} near-duplicate frames from {frames_dir.name}")
    return frames

log("✅ Showreel functions ready")



In [ ]:
# ==========================================
# 6. AUDIO & VIDEO ASSEMBLY
# ==========================================

def _ffmpeg_available() -> bool:
    if shutil.which("ffmpeg"):
        return True
    log("❌ FFmpeg not found in PATH")
    return False

def _log_ffmpeg_failure(label: str, result: subprocess.CompletedProcess) -> None:
    log(f"❌ FFmpeg {label} error (code {result.returncode})")
    stderr = (result.stderr or "").strip()
    if stderr:
        max_len = 2000
        snippet = stderr if len(stderr) <= max_len else stderr[-max_len:]
        log(snippet)

def _list_frame_files(frames_dir: Path) -> list:
    return sorted(frames_dir.glob("frame_*.png"))

def _frame_start_number(frame_files: list[Path]) -> int:
    for frame in frame_files:
        stem = frame.stem
        if not stem.startswith("frame_"):
            continue
        suffix = stem.split("_", 1)[1]
        if suffix.isdigit():
            return int(suffix)
    return 1

def add_audio(video_path: Path, audio_path: Path, output_path: Path, start_sec: float = 77) -> bool:
    """Mix audio starting at specified timestamp with video."""
    if not audio_path.exists():
        log(f"⚠️ Audio not found: {audio_path}")
        if video_path.exists() and video_path.stat().st_size > 0:
            shutil.copy(video_path, output_path)
            return True
        log(f"❌ Video not found to copy without audio: {video_path}")
        return False
    if not video_path.exists() or video_path.stat().st_size == 0:
        log(f"❌ Video not found for audio merge: {video_path}")
        return False
    if not _ffmpeg_available():
        return False

    cmd = [
        "ffmpeg", "-y",
        "-ss", str(float(start_sec)),
        "-i", str(audio_path),
        "-i", str(video_path),
        "-shortest",
        "-c:v", "copy",
        "-c:a", "aac",
        "-b:a", "192k",
        "-movflags", "+faststart",
        str(output_path)
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0 and output_path.exists() and output_path.stat().st_size > 0:
        log(f"✅ Audio added: {output_path}")
        return True
    _log_ffmpeg_failure("audio", result)
    fallback_cmd = [
        "ffmpeg", "-y",
        "-ss", str(float(start_sec)),
        "-i", str(audio_path),
        "-i", str(video_path),
        "-shortest",
        "-c:v", "copy",
        "-c:a", "libmp3lame",
        "-b:a", "192k",
        "-movflags", "+faststart",
        str(output_path)
    ]
    fallback = subprocess.run(fallback_cmd, capture_output=True, text=True)
    if fallback.returncode == 0 and output_path.exists() and output_path.stat().st_size > 0:
        log(f"✅ Audio added (mp3 fallback): {output_path}")
        return True
    _log_ffmpeg_failure("audio (mp3 fallback)", fallback)
    return False

def render_video_from_frames(frames_dir: Path, output_path: Path, fps: int = 24) -> bool:
    """Use FFmpeg to assemble frames into video."""
    if not _ffmpeg_available():
        return False
    if not frames_dir.exists():
        log(f"❌ Frames directory missing: {frames_dir}")
        return False
    frame_files = _list_frame_files(frames_dir)
    if not frame_files:
        log(f"❌ No frames found in {frames_dir}")
        return False
    start_number = _frame_start_number(frame_files)
    log(f"🎞️ Frames detected: {len(frame_files)} (start={start_number})")
    if frame_files[0].stat().st_size == 0:
        log(f"⚠️ First frame is empty: {frame_files[0]}")

    cmd = [
        "ffmpeg", "-y",
        "-framerate", str(fps),
        "-start_number", str(start_number),
        "-i", str(frames_dir / "frame_%06d.png"),
        "-vf", "scale=1080:1572:flags=lanczos,setsar=1",
        "-c:v", "libx264",
        "-profile:v", "main",
        "-level", "4.1",
        "-pix_fmt", "yuv420p",
        "-crf", "18",
        "-movflags", "+faststart",
        str(output_path)
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0 and output_path.exists() and output_path.stat().st_size > 0:
        size_kb = output_path.stat().st_size / 1024
        log(f"✅ Video rendered: {output_path} ({size_kb:.1f} KB)")
        return True
    _log_ffmpeg_failure("video", result)
    err = (result.stderr or "").lower()
    if "unknown encoder" in err or "libx264" in err:
        fallback_cmd = [
            "ffmpeg", "-y",
            "-framerate", str(fps),
            "-start_number", str(start_number),
            "-i", str(frames_dir / "frame_%06d.png"),
            "-vf", "scale=1080:1572:flags=lanczos,setsar=1",
            "-c:v", "mpeg4",
            "-q:v", "5",
            "-pix_fmt", "yuv420p",
            "-movflags", "+faststart",
            str(output_path)
        ]
        fallback = subprocess.run(fallback_cmd, capture_output=True, text=True)
        if fallback.returncode == 0 and output_path.exists() and output_path.stat().st_size > 0:
            size_kb = output_path.stat().st_size / 1024
            log(f"✅ Video rendered (mpeg4 fallback): {output_path} ({size_kb:.1f} KB)")
            return True
        _log_ffmpeg_failure("video (mpeg4 fallback)", fallback)
    return False

log("✅ Audio/Video functions ready")

In [ ]:
# ==========================================
# 7. ОСНОВНОЙ ПАЙПЛАЙН (Updated for Outro & Cities)
# ==========================================
import hashlib
import mimetypes
import requests
import time
import asyncio
import threading
# --- Poster overlay (Cyrillic; BebasNeue via TTF) ---
import sys
from pathlib import Path

# Ensure poster_overlay.py exists in /kaggle/working (Kaggle path rules can differ per runner).
def _ensure_poster_overlay_module():
    target = Path('/kaggle/working/poster_overlay.py')
    if target.exists():
        return
    try:
        code = 'from __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom pathlib import Path\nimport re\n\nimport cv2\nimport numpy as np\nfrom PIL import Image, ImageDraw, ImageFont, ImageFilter\n\n\n@dataclass(frozen=True)\nclass OverlayPlacement:\n    x: int\n    y: int\n    w: int\n    h: int\n    score: float\n\n\n_FONT_PATH: Path | None = None\n_FONT_LOGGED = False\n_MONTH_NAMES = (\n    "январ",\n    "феврал",\n    "март",\n    "апрел",\n    "мая",\n    "июн",\n    "июл",\n    "август",\n    "сентябр",\n    "октябр",\n    "ноябр",\n    "декабр",\n)\n\n\ndef _pick_font_path(*, search_roots: list[Path] | None = None) -> Path | None:\n    candidates: list[Path] = []\n    roots = list(search_roots or [])\n    roots.extend([Path.cwd(), Path("/kaggle/working")])\n\n    for root in roots:\n        for name in (\n            "BebasNeue-Bold.ttf",\n            "BebasNeue-Regular.ttf",\n            "Oswald-VariableFont_wght.ttf",\n        ):\n            p = root / name\n            if p.exists():\n                candidates.append(p)\n\n    inp = Path("/kaggle/input")\n    if inp.exists():\n        for pat in (\n            "*/BebasNeue-Bold.ttf",\n            "*/BebasNeue-Regular.ttf",\n            "*/assets/BebasNeue-Bold.ttf",\n            "*/assets/BebasNeue-Regular.ttf",\n            "*/Oswald-VariableFont_wght.ttf",\n            "*/assets/Oswald-VariableFont_wght.ttf",\n        ):\n            candidates.extend(inp.glob(pat))\n\n    # System fallback: guaranteed Cyrillic.\n    candidates.extend(\n        [\n            Path("/usr/share/fonts/truetype/dejavu/DejaVuSansCondensed.ttf"),\n            Path("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"),\n        ]\n    )\n\n    for p in candidates:\n        if p.exists():\n            return p\n    return None\n\n\ndef _font_supports_cyrillic(font: ImageFont.FreeTypeFont) -> bool:\n    for ch in ("Я", "Ж", "Ю", "П"):\n        try:\n            bbox = font.getbbox(ch)\n        except Exception:\n            return False\n        if not bbox or (bbox[2] - bbox[0]) <= 0:\n            return False\n    return True\n\n\ndef _load_font(size: int, *, search_roots: list[Path] | None = None) -> ImageFont.FreeTypeFont:\n    global _FONT_PATH\n    if _FONT_PATH is None:\n        _FONT_PATH = _pick_font_path(search_roots=search_roots)\n\n    if _FONT_PATH is not None:\n        try:\n            f = ImageFont.truetype(str(_FONT_PATH), int(size))\n            if _font_supports_cyrillic(f):\n                return f\n        except Exception:\n            pass\n\n    # Hard fallback.\n    for p in (\n        Path("/usr/share/fonts/truetype/dejavu/DejaVuSansCondensed.ttf"),\n        Path("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"),\n    ):\n        if p.exists():\n            return ImageFont.truetype(str(p), int(size))\n    return ImageFont.load_default()\n\n\ndef _wrap_line(\n    draw: ImageDraw.ImageDraw,\n    text: str,\n    font: ImageFont.FreeTypeFont,\n    max_w: int,\n    *,\n    max_lines: int,\n) -> list[str]:\n    words = [w for w in (text or "").split(" ") if w]\n    out: list[str] = []\n    cur = ""\n    for w in words:\n        cand = (cur + " " + w).strip() if cur else w\n        bbox = draw.textbbox((0, 0), cand, font=font)\n        if (bbox[2] - bbox[0]) <= max_w:\n            cur = cand\n            continue\n        if cur:\n            out.append(cur)\n            if len(out) >= max_lines:\n                return out\n        cur = w\n    if cur:\n        out.append(cur)\n    return out[:max_lines]\n\n\ndef _looks_like_fact_line(text: str) -> bool:\n    lowered = (text or "").strip().casefold()\n    if not lowered:\n        return False\n    if "," in lowered:\n        return True\n    if re.search(r"\\b\\d{1,2}[:.]\\d{2}\\b", lowered):\n        return True\n    if any(month in lowered for month in _MONTH_NAMES) and re.search(r"\\b\\d{1,2}\\b", lowered):\n        return True\n    return False\n\n\ndef _edge_density(gray: np.ndarray, x0: int, y0: int, bw: int, bh: int) -> float:\n    edges = cv2.Canny(gray, 60, 160)\n    edges = cv2.dilate(edges, cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3)), iterations=1)\n    roi = edges[y0 : y0 + bh, x0 : x0 + bw]\n    return float(np.mean(roi > 0))\n\n\ndef _build_text_mask(gray: np.ndarray) -> np.ndarray:\n    h, w = gray.shape[:2]\n    base = float(min(w, h))\n    blur = cv2.GaussianBlur(gray, (5, 5), 0)\n    adaptive = cv2.adaptiveThreshold(\n        blur,\n        255,\n        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,\n        cv2.THRESH_BINARY_INV,\n        31,\n        11,\n    )\n    gradient = cv2.morphologyEx(\n        blur,\n        cv2.MORPH_GRADIENT,\n        cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3)),\n    )\n    _, gradient_bin = cv2.threshold(\n        gradient, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU\n    )\n    edges = cv2.Canny(blur, 50, 150)\n    mask = cv2.bitwise_or(adaptive, gradient_bin)\n    mask = cv2.bitwise_or(mask, edges)\n    close_kernel = cv2.getStructuringElement(\n        cv2.MORPH_RECT,\n        (\n            max(3, int(round(base * 0.010))),\n            max(3, int(round(base * 0.006))),\n        ),\n    )\n    expand_kernel = cv2.getStructuringElement(\n        cv2.MORPH_RECT,\n        (\n            max(15, int(round(base * 0.032))),\n            max(9, int(round(base * 0.020))),\n        ),\n    )\n    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, close_kernel, iterations=1)\n    mask = cv2.dilate(mask, expand_kernel, iterations=1)\n    return mask\n\n\ndef _mask_fill_ratio(mask_integral: np.ndarray, x0: int, y0: int, bw: int, bh: int) -> float:\n    x1 = x0 + bw\n    y1 = y0 + bh\n    filled = (\n        int(mask_integral[y1, x1])\n        - int(mask_integral[y0, x1])\n        - int(mask_integral[y1, x0])\n        + int(mask_integral[y0, x0])\n    )\n    return filled / float(max(bw * bh, 1))\n\n\ndef _find_best_placement(img_bgr: np.ndarray, box_w: int, box_h: int) -> OverlayPlacement:\n    h, w = img_bgr.shape[:2]\n    base = float(min(w, h))\n    margin = int(max(18, min(42, base * 0.024)))\n    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)\n    text_mask = _build_text_mask(gray)\n    mask_integral = cv2.integral((text_mask > 0).astype(np.uint8), sdepth=cv2.CV_32S)\n\n    max_x = max(margin, w - box_w - margin)\n    max_y = max(margin, h - box_h - margin)\n    step_x = max(margin, box_w // 4)\n    step_y = max(margin, box_h // 4)\n\n    x_positions = set(range(margin, max_x + 1, step_x))\n    y_positions = set(range(margin, max_y + 1, step_y))\n    x_positions.update({margin, max_x, max(margin, (w - box_w) // 2)})\n    y_positions.update({margin, max_y, max(margin, (h - box_h) // 2), max(margin, int(h * 0.62))})\n\n    best: OverlayPlacement | None = None\n    for y0 in sorted(y_positions):\n        for x0 in sorted(x_positions):\n            x0 = int(max(margin, min(max_x, x0)))\n            y0 = int(max(margin, min(max_y, y0)))\n            fill_ratio = _mask_fill_ratio(mask_integral, x0, y0, box_w, box_h)\n            edge_score = _edge_density(gray, x0, y0, box_w, box_h)\n            variance_score = float(np.std(gray[y0 : y0 + box_h, x0 : x0 + box_w])) / 255.0\n            center_x = (x0 + (box_w / 2.0)) / float(max(w, 1))\n            center_y = (y0 + (box_h / 2.0)) / float(max(h, 1))\n            preference = abs(center_y - 0.72) * 0.10 + (0.5 - abs(center_x - 0.5)) * 0.05\n            hard_overlap_penalty = 0.0\n            if fill_ratio > 0.015:\n                hard_overlap_penalty = fill_ratio * 40.0\n            score = (\n                fill_ratio * 12.0\n                + hard_overlap_penalty\n                + edge_score * 2.5\n                + variance_score * 0.8\n                + preference\n            )\n            placement = OverlayPlacement(x=x0, y=y0, w=box_w, h=box_h, score=score)\n            if best is None or placement.score < best.score:\n                best = placement\n\n    if best is None:\n        return OverlayPlacement(\n            x=margin,\n            y=h - box_h - margin,\n            w=box_w,\n            h=box_h,\n            score=1e9,\n        )\n    return best\n\n\ndef apply_poster_overlay(\n    input_path: str | Path,\n    *,\n    text: str,\n    out_dir: str | Path,\n    search_roots: list[str | Path] | None = None,\n    highlight_title: bool | None = None,\n) -> Path:\n    """Draw a modern badge using a Cyrillic-capable TTF font (BebasNeue if available)."""\n\n    global _FONT_LOGGED\n\n    in_path = Path(input_path)\n    text = (text or "").strip()\n    if not text:\n        return in_path\n\n    bgr = cv2.imread(str(in_path), cv2.IMREAD_COLOR)\n    if bgr is None:\n        return in_path\n\n    h, w = bgr.shape[:2]\n    base = float(min(w, h))\n    roots: list[Path] = []\n    if search_roots:\n        for r in search_roots:\n            try:\n                roots.append(Path(r))\n            except Exception:\n                pass\n\n    # Typography\n    title_size = int(max(42, min(98, base * 0.065)))\n    body_size = int(max(30, min(68, base * 0.046)))\n    font_title = _load_font(title_size, search_roots=roots)\n    font_body = _load_font(body_size, search_roots=roots)\n\n    if not _FONT_LOGGED:\n        _FONT_LOGGED = True\n        print(f"✅ Overlay font: {_FONT_PATH} (title_ok={_font_supports_cyrillic(font_title)} body_ok={_font_supports_cyrillic(font_body)})")\n\n    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)\n    img = Image.fromarray(rgb).convert("RGBA")\n    draw = ImageDraw.Draw(img)\n\n    raw = [" ".join(l.split()).strip() for l in text.splitlines() if " ".join(l.split()).strip()]\n    if not raw:\n        return in_path\n\n    if highlight_title is None:\n        highlight_title = len(raw) > 1 and not _looks_like_fact_line(raw[0])\n\n    max_box_w = int(w * (0.66 if highlight_title else 0.54))\n    pad_x = int(max(36, min(78, base * 0.050)))\n    pad_y = int(max(26, min(64, base * 0.040)))\n    max_text_w = max_box_w - pad_x * 2\n\n    # Title can wrap to 2 lines; fact-only overlays use a more compact body-only layout.\n    lines: list[tuple[str, ImageFont.FreeTypeFont]] = []\n    remaining = list(raw)\n    body_wrap_lines = 2 if highlight_title else 3\n\n    if highlight_title and remaining:\n        for part in _wrap_line(draw, remaining[0], font_title, max_text_w, max_lines=2):\n            lines.append((part, font_title))\n            if len(lines) >= 6:\n                break\n        remaining = remaining[1:]\n\n    for extra in remaining:\n        for part in _wrap_line(draw, extra, font_body, max_text_w, max_lines=body_wrap_lines):\n            lines.append((part, font_body))\n            if len(lines) >= 6:\n                break\n        if len(lines) >= 6:\n            break\n\n    stroke_w = 2\n    line_boxes = [draw.textbbox((0, 0), t, font=f, stroke_width=stroke_w) for t, f in lines]\n    line_heights = [b[3] - b[1] for b in line_boxes]\n    line_widths = [b[2] - b[0] for b in line_boxes]\n    gap = int(max(10, min(22, base * 0.014)))\n    text_h = sum(line_heights) + gap * (len(lines) - 1)\n    box_w = min(max_box_w, max(line_widths) + pad_x * 2)\n    box_h = min(int(h * 0.44), text_h + pad_y * 2)\n\n    placement = _find_best_placement(bgr, box_w, box_h)\n    x0, y0 = placement.x, placement.y\n\n    radius = int(max(16, min(40, base * 0.03)))\n    border = max(2, int(round(base * 0.0022)))\n    shadow_off = int(max(6, min(14, base * 0.01)))\n    shadow_blur = int(max(10, min(28, base * 0.02)))\n\n    shadow = Image.new("RGBA", img.size, (0, 0, 0, 0))\n    sd = ImageDraw.Draw(shadow)\n    sd.rounded_rectangle(\n        [x0 + shadow_off, y0 + shadow_off, x0 + box_w + shadow_off, y0 + box_h + shadow_off],\n        radius=radius,\n        fill=(0, 0, 0, 110),\n    )\n    shadow = shadow.filter(ImageFilter.GaussianBlur(radius=shadow_blur))\n    img = Image.alpha_composite(img, shadow)\n\n    panel = Image.new("RGBA", img.size, (0, 0, 0, 0))\n    pd = ImageDraw.Draw(panel)\n    pd.rounded_rectangle([x0, y0, x0 + box_w, y0 + box_h], radius=radius, fill=(12, 12, 14, 215))\n    pd.rounded_rectangle(\n        [x0, y0, x0 + box_w, y0 + box_h],\n        radius=radius,\n        outline=(255, 255, 255, 150),\n        width=border,\n    )\n    img = Image.alpha_composite(img, panel)\n\n    draw = ImageDraw.Draw(img)\n    tx = x0 + pad_x\n    ty = y0 + pad_y\n    for (t, f), lh in zip(lines, line_heights):\n        draw.text(\n            (tx, ty),\n            t,\n            font=f,\n            fill=(255, 255, 255, 255),\n            stroke_width=stroke_w,\n            stroke_fill=(0, 0, 0, 140),\n        )\n        ty += lh + gap\n\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    out_path = out_dir / f"{in_path.stem}__overlay.png"\n    img.convert("RGB").save(out_path, format="PNG")\n    return out_path'
        target.write_text(code, encoding='utf-8')
        print(f'✅ Wrote poster_overlay.py to {target}')
    except Exception as e:
        print(f'⚠️ Failed to write poster_overlay.py: {e}')

_ensure_poster_overlay_module()

def _ensure_story_publish_module():
    target = Path('/kaggle/working/story_publish.py')
    if target.exists():
        return
    try:
        code = 'from __future__ import annotations\n\nimport json\nimport logging\nimport mimetypes\nfrom fractions import Fraction\nfrom pathlib import Path\nimport shutil\nimport subprocess\nfrom typing import Any\n\nimport cv2\nimport requests\nfrom cryptography.fernet import Fernet\nfrom telethon import TelegramClient, functions, types\nfrom telethon.sessions import StringSession\n\n\nlogger = logging.getLogger(__name__)\n\nCONFIG_FILENAME = "story_publish.json"\nCIPHER_FILENAME = "story_publish.enc"\nKEY_FILENAME = "story_publish.key"\nMAX_VIDEO_BYTES = 30 * 1024 * 1024\nPREMIUM_REQUIRED_ERROR_NAME = "PremiumAccountRequiredError"\nCRUMPLE_VIDEO_STORY_PREVIEW_TS = 0\nSTORY_VIDEO_WIDTH = 720\nSTORY_VIDEO_HEIGHT = 1280\nSTORY_SAFE_VIDEO_FILENAME = "crumple_video_story_720x1280.mp4"\nSTORY_VIDEO_PRESET = "fast"\nSTORY_VIDEO_BITRATE = "900k"\nSTORY_VIDEO_MAXRATE = "1200k"\nSTORY_VIDEO_BUFSIZE = "2400k"\nSTORY_UPLOAD_PROFILE_LEGACY_H264 = "legacy_h264_transcode"\nSTORY_UPLOAD_PROFILE_NATIVE_HEVC = "telegram_story_native_hevc_720p_v1"\nCHERRYFLASH_NATIVE_TARGET_BYTES = 15 * 1024 * 1024\nCHERRYFLASH_NATIVE_VIDEO_TAG = "hvc1"\nCHERRYFLASH_NATIVE_VIDEO_CODEC = "hevc"\nCHERRYFLASH_NATIVE_AUDIO_CODEC = "aac"\nCHERRYFLASH_NATIVE_AUDIO_SAMPLE_RATE = 48000\nCHERRYFLASH_NATIVE_AUDIO_CHANNELS = 2\nCHERRYFLASH_NATIVE_MAX_DURATION_SECONDS = 60.0\n\n\ndef _find_input_file(filename: str, *, search_roots: list[Path]) -> Path | None:\n    seen: set[Path] = set()\n    for root in search_roots:\n        candidate = root / filename\n        if candidate.exists():\n            return candidate\n        if root in seen or not root.exists():\n            continue\n        seen.add(root)\n        try:\n            matches = sorted(root.rglob(filename))\n        except Exception:\n            matches = []\n        if matches:\n            return matches[0]\n    return None\n\n\ndef load_story_publish_runtime(*, search_roots: list[Path], log) -> dict[str, Any] | None:\n    config_path = _find_input_file(CONFIG_FILENAME, search_roots=search_roots)\n    if not config_path:\n        log("[SKIP] story_publish.json not found; story publish disabled for this run.")\n        return None\n\n    cipher_path = _find_input_file(CIPHER_FILENAME, search_roots=search_roots)\n    key_path = _find_input_file(KEY_FILENAME, search_roots=search_roots)\n    if not cipher_path or not key_path:\n        raise RuntimeError("Story publish config exists but encrypted secrets are missing")\n\n    config = json.loads(config_path.read_text(encoding="utf-8"))\n    if not isinstance(config, dict):\n        raise RuntimeError("story_publish.json must be an object")\n    if not isinstance(config.get("targets"), list) or not config["targets"]:\n        raise RuntimeError("story_publish.json must contain non-empty targets")\n\n    fernet = Fernet(key_path.read_bytes().strip())\n    auth_raw = fernet.decrypt(cipher_path.read_bytes())\n    auth = json.loads(auth_raw.decode("utf-8"))\n    if not isinstance(auth, dict):\n        raise RuntimeError("Decrypted story auth payload must be an object")\n    return {"config": config, "auth": auth}\n\n\ndef _story_media_path(\n    config: dict[str, Any],\n    *,\n    final_video_path: Path | None,\n    intro_path: Path | None,\n    posters: list[Path] | None,\n) -> Path:\n    mode = str(config.get("mode") or "video").strip().lower()\n    if mode == "image":\n        for candidate in list(posters or [])[1:]:\n            if candidate and candidate.exists():\n                return candidate\n        if intro_path and intro_path.exists():\n            return intro_path\n        for candidate in posters or []:\n            if candidate and candidate.exists():\n                return candidate\n        raise RuntimeError("Image smoke mode requested but no image candidate was found")\n\n    if not final_video_path or not final_video_path.exists():\n        raise RuntimeError("Story video publish requested but final video is missing")\n    return final_video_path\n\n\ndef _ffmpeg_available() -> bool:\n    return shutil.which("ffmpeg") is not None\n\n\ndef _ffprobe_available() -> bool:\n    return shutil.which("ffprobe") is not None\n\n\ndef _video_dimensions(path: Path) -> tuple[int, int] | None:\n    cap = cv2.VideoCapture(str(path))\n    if not cap.isOpened():\n        return None\n    try:\n        width = max(1, int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or STORY_VIDEO_WIDTH))\n        height = max(1, int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or STORY_VIDEO_HEIGHT))\n        return width, height\n    finally:\n        cap.release()\n\n\ndef _ffprobe_json(path: Path) -> dict[str, Any] | None:\n    if not _ffprobe_available():\n        return None\n    result = subprocess.run(\n        [\n            "ffprobe",\n            "-v",\n            "error",\n            "-show_format",\n            "-show_streams",\n            "-print_format",\n            "json",\n            str(path),\n        ],\n        capture_output=True,\n        text=True,\n        check=False,\n    )\n    if result.returncode != 0:\n        return None\n    try:\n        data = json.loads(result.stdout or "{}")\n    except Exception:\n        return None\n    return data if isinstance(data, dict) else None\n\n\ndef _fps_from_ratio(value: str | None) -> float | None:\n    raw = str(value or "").strip()\n    if not raw or raw == "0/0":\n        return None\n    try:\n        return float(Fraction(raw))\n    except Exception:\n        return None\n\n\ndef _video_probe(path: Path) -> dict[str, Any]:\n    info: dict[str, Any] = {"path": str(path)}\n    try:\n        info["size_bytes"] = int(path.stat().st_size)\n    except Exception:\n        pass\n    cap = cv2.VideoCapture(str(path))\n    if not cap.isOpened():\n        info["readable"] = False\n        return info\n    try:\n        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)\n        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)\n        fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)\n        frame_count = float(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0.0)\n        duration = frame_count / fps if fps > 0 and frame_count > 0 else 0.0\n        info.update(\n            {\n                "readable": True,\n                "width": width,\n                "height": height,\n                "fps": round(fps, 3),\n                "frame_count": int(frame_count) if frame_count > 0 else 0,\n                "duration_seconds": round(duration, 3),\n            }\n        )\n        size_bytes = info.get("size_bytes")\n        if duration > 0 and isinstance(size_bytes, int):\n            info["approx_bitrate_kbps"] = round(size_bytes * 8 / duration / 1000, 1)\n    finally:\n        cap.release()\n    ffprobe_data = _ffprobe_json(path)\n    if isinstance(ffprobe_data, dict):\n        format_info = ffprobe_data.get("format")\n        if isinstance(format_info, dict):\n            format_name = format_info.get("format_name")\n            if format_name:\n                info["format_name"] = str(format_name)\n            format_bit_rate = format_info.get("bit_rate")\n            if format_bit_rate:\n                try:\n                    info["format_bit_rate"] = int(format_bit_rate)\n                except Exception:\n                    pass\n        streams = ffprobe_data.get("streams")\n        if isinstance(streams, list):\n            video_stream = next(\n                (stream for stream in streams if isinstance(stream, dict) and stream.get("codec_type") == "video"),\n                None,\n            )\n            audio_stream = next(\n                (stream for stream in streams if isinstance(stream, dict) and stream.get("codec_type") == "audio"),\n                None,\n            )\n            if isinstance(video_stream, dict):\n                codec_name = video_stream.get("codec_name")\n                if codec_name:\n                    info["video_codec"] = str(codec_name)\n                pix_fmt = video_stream.get("pix_fmt")\n                if pix_fmt:\n                    info["pix_fmt"] = str(pix_fmt)\n                codec_tag = video_stream.get("codec_tag_string")\n                if codec_tag:\n                    info["video_tag"] = str(codec_tag)\n                ffprobe_fps = _fps_from_ratio(\n                    str(video_stream.get("avg_frame_rate") or video_stream.get("r_frame_rate") or "")\n                )\n                if ffprobe_fps is not None:\n                    info["fps"] = round(ffprobe_fps, 3)\n            if isinstance(audio_stream, dict):\n                audio_codec = audio_stream.get("codec_name")\n                if audio_codec:\n                    info["audio_codec"] = str(audio_codec)\n                sample_rate = audio_stream.get("sample_rate")\n                if sample_rate:\n                    try:\n                        info["audio_sample_rate"] = int(sample_rate)\n                    except Exception:\n                        pass\n                channels = audio_stream.get("channels")\n                if channels:\n                    try:\n                        info["audio_channels"] = int(channels)\n                    except Exception:\n                        pass\n    return info\n\n\ndef _story_upload_profile(config: dict[str, Any]) -> str:\n    raw = str(config.get("upload_profile") or "").strip().lower()\n    if raw == STORY_UPLOAD_PROFILE_NATIVE_HEVC:\n        return STORY_UPLOAD_PROFILE_NATIVE_HEVC\n    return STORY_UPLOAD_PROFILE_LEGACY_H264\n\n\ndef _validate_native_story_video(path: Path, *, log) -> Path:\n    if not path.exists():\n        raise RuntimeError("Story video publish requested but final video is missing")\n    probe = _video_probe(path)\n    problems: list[str] = []\n    if int(probe.get("size_bytes") or 0) > CHERRYFLASH_NATIVE_TARGET_BYTES:\n        problems.append(\n            f"size={probe.get(\'size_bytes\')} exceeds {CHERRYFLASH_NATIVE_TARGET_BYTES} bytes"\n        )\n    if int(probe.get("width") or 0) != STORY_VIDEO_WIDTH or int(probe.get("height") or 0) != STORY_VIDEO_HEIGHT:\n        problems.append(\n            f"canvas={probe.get(\'width\')}x{probe.get(\'height\')} expected {STORY_VIDEO_WIDTH}x{STORY_VIDEO_HEIGHT}"\n        )\n    fps = probe.get("fps")\n    if not isinstance(fps, (int, float)) or abs(float(fps) - 30.0) > 0.2:\n        problems.append(f"fps={fps!r} expected ~30")\n    duration_seconds = probe.get("duration_seconds")\n    if isinstance(duration_seconds, (int, float)) and float(duration_seconds) > CHERRYFLASH_NATIVE_MAX_DURATION_SECONDS:\n        problems.append(\n            f"duration_seconds={duration_seconds} exceeds {CHERRYFLASH_NATIVE_MAX_DURATION_SECONDS}"\n        )\n    if str(probe.get("format_name") or "").find("mp4") < 0:\n        problems.append(f"format={probe.get(\'format_name\')!r} is not mp4")\n    if str(probe.get("video_codec") or "").strip().lower() != CHERRYFLASH_NATIVE_VIDEO_CODEC:\n        problems.append(f"video_codec={probe.get(\'video_codec\')!r} expected {CHERRYFLASH_NATIVE_VIDEO_CODEC}")\n    if str(probe.get("video_tag") or "").strip().lower() != CHERRYFLASH_NATIVE_VIDEO_TAG:\n        problems.append(f"video_tag={probe.get(\'video_tag\')!r} expected {CHERRYFLASH_NATIVE_VIDEO_TAG}")\n    if str(probe.get("pix_fmt") or "").strip().lower() != "yuv420p":\n        problems.append(f"pix_fmt={probe.get(\'pix_fmt\')!r} expected \'yuv420p\'")\n    if str(probe.get("audio_codec") or "").strip().lower() != CHERRYFLASH_NATIVE_AUDIO_CODEC:\n        problems.append(f"audio_codec={probe.get(\'audio_codec\')!r} expected {CHERRYFLASH_NATIVE_AUDIO_CODEC}")\n    if int(probe.get("audio_sample_rate") or 0) != CHERRYFLASH_NATIVE_AUDIO_SAMPLE_RATE:\n        problems.append(\n            f"audio_sample_rate={probe.get(\'audio_sample_rate\')!r} expected {CHERRYFLASH_NATIVE_AUDIO_SAMPLE_RATE}"\n        )\n    if int(probe.get("audio_channels") or 0) != CHERRYFLASH_NATIVE_AUDIO_CHANNELS:\n        problems.append(\n            f"audio_channels={probe.get(\'audio_channels\')!r} expected {CHERRYFLASH_NATIVE_AUDIO_CHANNELS}"\n        )\n    if problems:\n        raise RuntimeError(\n            "CherryFlash story upload requires a native Telegram-ready final mp4; "\n            + "; ".join(problems)\n        )\n    story_kb = int(probe.get("size_bytes") or path.stat().st_size) / 1024\n    bitrate_label = ""\n    if isinstance(probe.get("approx_bitrate_kbps"), (int, float)):\n        bitrate_label = f", ~{probe[\'approx_bitrate_kbps\']} kbps"\n    log(\n        "✅ Story-native video validated: "\n        f"{path.name} ({probe.get(\'width\')}x{probe.get(\'height\')}, "\n        f"{probe.get(\'video_codec\')}/{probe.get(\'audio_codec\')}, "\n        f"{story_kb:.1f} KiB{bitrate_label})"\n    )\n    return path\n\n\ndef _ensure_story_safe_video(\n    path: Path,\n    *,\n    output_dir: Path,\n    log,\n) -> Path:\n    dimensions = _video_dimensions(path)\n    if not _ffmpeg_available():\n        raise RuntimeError("ffmpeg is required to prepare story-safe video upload")\n\n    output_dir.mkdir(parents=True, exist_ok=True)\n    story_path = output_dir / STORY_SAFE_VIDEO_FILENAME\n    if dimensions == (STORY_VIDEO_WIDTH, STORY_VIDEO_HEIGHT):\n        filter_graph = "setsar=1"\n    else:\n        filter_graph = (\n            f"scale={STORY_VIDEO_WIDTH}:{STORY_VIDEO_HEIGHT}:"\n            "force_original_aspect_ratio=decrease:flags=lanczos,"\n            f"pad={STORY_VIDEO_WIDTH}:{STORY_VIDEO_HEIGHT}:(ow-iw)/2:(oh-ih)/2:color=black,"\n            "setsar=1"\n        )\n    result = subprocess.run(\n        [\n            "ffmpeg",\n            "-y",\n            "-i",\n            str(path),\n            "-vf",\n            filter_graph,\n            "-r",\n            "30",\n            "-map",\n            "0:v:0",\n            "-map",\n            "0:a?",\n            "-c:v",\n            "libx264",\n            "-profile:v",\n            "high",\n            "-level:v",\n            "4.1",\n            "-preset",\n            STORY_VIDEO_PRESET,\n            "-b:v",\n            STORY_VIDEO_BITRATE,\n            "-maxrate",\n            STORY_VIDEO_MAXRATE,\n            "-bufsize",\n            STORY_VIDEO_BUFSIZE,\n            "-pix_fmt",\n            "yuv420p",\n            "-g",\n            "30",\n            "-keyint_min",\n            "30",\n            "-sc_threshold",\n            "0",\n            "-bf",\n            "0",\n            "-tag:v",\n            "avc1",\n            "-movflags",\n            "+faststart",\n            "-c:a",\n            "aac",\n            "-b:a",\n            "128k",\n            "-ac",\n            "2",\n            "-ar",\n            "44100",\n            "-shortest",\n            str(story_path),\n        ],\n        capture_output=True,\n        text=True,\n        check=False,\n    )\n    if result.returncode != 0:\n        stderr = (result.stderr or "").strip()\n        raise RuntimeError(\n            "Failed to prepare story-safe video canvas"\n            + (f": {stderr}" if stderr else "")\n        )\n    output_dimensions = _video_dimensions(story_path)\n    if output_dimensions != (STORY_VIDEO_WIDTH, STORY_VIDEO_HEIGHT):\n        raise RuntimeError(\n            "Story-safe video has unexpected canvas "\n            f"{output_dimensions!r}, expected {(STORY_VIDEO_WIDTH, STORY_VIDEO_HEIGHT)!r}"\n        )\n    if story_path.stat().st_size > MAX_VIDEO_BYTES:\n        raise RuntimeError(\n            "Story-safe video exceeds Telegram story limit "\n            f"({story_path.stat().st_size} bytes > {MAX_VIDEO_BYTES})"\n        )\n    original_label = (\n        f"{dimensions[0]}x{dimensions[1]}"\n        if isinstance(dimensions, tuple) and len(dimensions) == 2\n        else "unknown"\n    )\n    story_info = _video_probe(story_path)\n    story_size = int(story_info.get("size_bytes") or story_path.stat().st_size)\n    story_kb = story_size / 1024\n    story_bitrate = story_info.get("approx_bitrate_kbps")\n    bitrate_label = (\n        f", ~{story_bitrate} kbps" if isinstance(story_bitrate, (int, float)) else ""\n    )\n    log(\n        "✅ Story-safe video prepared: "\n        f"{story_path.name} ({original_label} -> {STORY_VIDEO_WIDTH}x{STORY_VIDEO_HEIGHT}, "\n        f"h264/aac, bitrate={STORY_VIDEO_BITRATE}, maxrate={STORY_VIDEO_MAXRATE}, "\n        f"{story_kb:.1f} KiB{bitrate_label})"\n    )\n    return story_path\n\n\nasync def _create_client(auth: dict[str, Any]) -> TelegramClient:\n    client = TelegramClient(\n        StringSession(str(auth.get("session") or "").strip()),\n        int(auth["api_id"]),\n        str(auth["api_hash"]),\n        device_model=str(auth.get("device_model") or "iPhone 14 Pro"),\n        system_version=str(auth.get("system_version") or "iOS 17.2"),\n        app_version=str(auth.get("app_version") or "10.5.1"),\n        lang_code=str(auth.get("lang_code") or "ru"),\n        system_lang_code=str(auth.get("system_lang_code") or auth.get("lang_code") or "ru"),\n    )\n    await client.connect()\n    if not await client.is_user_authorized():\n        await client.disconnect()\n        raise RuntimeError("Story publish Telethon client is not authorized")\n    return client\n\n\ndef _account_info(me: Any) -> dict[str, Any]:\n    return {\n        "id": getattr(me, "id", None),\n        "username": getattr(me, "username", None),\n        "premium": bool(getattr(me, "premium", False)),\n    }\n\n\ndef _account_label(account: dict[str, Any] | None) -> str:\n    if not isinstance(account, dict):\n        return "current account"\n    username = str(account.get("username") or "").strip()\n    if username:\n        return f"@{username}"\n    account_id = account.get("id")\n    if account_id:\n        return f"id={account_id}"\n    return "current account"\n\n\ndef _format_story_error(exc: Exception, *, account: dict[str, Any] | None) -> str:\n    base = f"{type(exc).__name__}: {exc}"\n    if type(exc).__name__ == PREMIUM_REQUIRED_ERROR_NAME and account and not account.get("premium"):\n        return (\n            f"{base}. {_account_label(account)} is not Telegram Premium, "\n            "so this session cannot publish stories."\n        )\n    return base\n\n\ndef _video_attributes(path: Path) -> list[types.TypeDocumentAttribute]:\n    cap = cv2.VideoCapture(str(path))\n    if not cap.isOpened():\n        return [\n            types.DocumentAttributeVideo(\n                duration=1.0,\n                w=1080,\n                h=1920,\n                supports_streaming=True,\n            )\n        ]\n    try:\n        width = max(1, int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 1080))\n        height = max(1, int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 1920))\n        fps = float(cap.get(cv2.CAP_PROP_FPS) or 24.0)\n        frame_count = float(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0.0)\n        duration = frame_count / fps if fps > 0 and frame_count > 0 else 1.0\n    finally:\n        cap.release()\n    return [\n        types.DocumentAttributeVideo(\n            duration=max(1.0, float(duration)),\n            w=width,\n            h=height,\n            supports_streaming=True,\n        )\n    ]\n\n\nasync def _input_media_for_path(client: TelegramClient, path: Path):\n    suffix = path.suffix.lower()\n    uploaded = await client.upload_file(str(path))\n    if suffix in {".jpg", ".jpeg", ".png", ".webp"}:\n        return types.InputMediaUploadedPhoto(file=uploaded)\n    if suffix in {".mp4", ".mov", ".mkv", ".webm"}:\n        if path.stat().st_size > MAX_VIDEO_BYTES:\n            raise RuntimeError(\n                f"Story video exceeds Telegram story limit ({path.stat().st_size} bytes > {MAX_VIDEO_BYTES})"\n            )\n        mime_type = mimetypes.guess_type(str(path))[0] or "video/mp4"\n        return types.InputMediaUploadedDocument(\n            file=uploaded,\n            mime_type=mime_type,\n            attributes=_video_attributes(path),\n            video_timestamp=CRUMPLE_VIDEO_STORY_PREVIEW_TS,\n        )\n    raise RuntimeError(f"Unsupported story media type: {path.name}")\n\n\ndef _extract_story_id(result: Any) -> int | None:\n    updates = getattr(result, "updates", None) or []\n    for update in updates:\n        if isinstance(update, types.UpdateStoryID):\n            return int(update.id)\n        if isinstance(update, types.UpdateStory):\n            story = getattr(update, "story", None)\n            story_id = getattr(story, "id", None)\n            if story_id is not None:\n                return int(story_id)\n    return None\n\n\ndef _json_default(value: Any) -> Any:\n    if isinstance(value, Path):\n        return str(value)\n    isoformat = getattr(value, "isoformat", None)\n    if callable(isoformat):\n        try:\n            return isoformat()\n        except Exception:\n            pass\n    return str(value)\n\n\ndef write_story_publish_report(report: dict[str, Any], *, output_dir: Path) -> Path:\n    output_dir.mkdir(parents=True, exist_ok=True)\n    report_path = output_dir / "story_publish_report.json"\n    report_path.write_text(\n        json.dumps(report, ensure_ascii=False, indent=2, default=_json_default),\n        encoding="utf-8",\n    )\n    return report_path\n\n\ndef _build_story_report(\n    config: dict[str, Any],\n    *,\n    phase: str,\n    account: dict[str, Any],\n    media_path: Path | None = None,\n) -> dict[str, Any]:\n    report: dict[str, Any] = {\n        "ok": False,\n        "phase": phase,\n        "mode": config.get("mode") or "video",\n        "period_seconds": int(config.get("period_seconds") or 24 * 60 * 60),\n        "pinned": bool(config.get("pinned")),\n        "account": account,\n        "targets": [],\n        "blocking_ok": False,\n        "required_ok": False,\n        "fanout_ok": False,\n        "partial_ok": False,\n    }\n    if media_path is not None:\n        report["media_path"] = str(media_path)\n        report["media"] = _video_probe(media_path)\n    return report\n\n\ndef _story_target_is_blocking(target_cfg: dict[str, Any], *, index: int) -> bool:\n    blocking = target_cfg.get("blocking")\n    if isinstance(blocking, bool):\n        return blocking\n    return index == 0\n\n\ndef _story_target_is_required(target_cfg: dict[str, Any], *, blocking: bool) -> bool:\n    required = target_cfg.get("required")\n    if isinstance(required, bool):\n        return required\n    return blocking\n\n\ndef _business_connection_for_target(\n    auth: dict[str, Any],\n    target_cfg: dict[str, Any],\n) -> dict[str, Any] | None:\n    target_hash = str(target_cfg.get("business_connection_hash") or "").strip()\n    for item in auth.get("business_connections") or []:\n        if not isinstance(item, dict):\n            continue\n        if str(item.get("connection_hash") or "").strip() == target_hash:\n            return item\n    return None\n\n\ndef _business_story_result_id(result: dict[str, Any]) -> int | None:\n    payload = result.get("result")\n    if not isinstance(payload, dict):\n        return None\n    story_id = payload.get("id")\n    try:\n        return int(story_id)\n    except (TypeError, ValueError):\n        return None\n\n\ndef _post_business_story(\n    *,\n    auth: dict[str, Any],\n    target_cfg: dict[str, Any],\n    media_path: Path,\n    config: dict[str, Any],\n) -> dict[str, Any]:\n    connection = _business_connection_for_target(auth, target_cfg)\n    if not connection:\n        raise RuntimeError("business connection secret is missing")\n    if not bool(connection.get("is_enabled")):\n        raise RuntimeError("business connection is disabled")\n    if not bool(connection.get("can_manage_stories")):\n        raise RuntimeError("business connection lacks can_manage_stories")\n    bot_token = str(auth.get("business_bot_token") or "").strip()\n    if not bot_token:\n        raise RuntimeError("business Bot API token is missing")\n    connection_id = str(connection.get("connection_id") or "").strip()\n    if not connection_id:\n        raise RuntimeError("business connection id is missing")\n\n    suffix = media_path.suffix.lower()\n    attach_name = "story"\n    if suffix in {".jpg", ".jpeg", ".png", ".webp"}:\n        content = {"type": "photo", "photo": f"attach://{attach_name}"}\n    elif suffix in {".mp4", ".mov"}:\n        content = {\n            "type": "video",\n            "video": f"attach://{attach_name}",\n            "cover_frame_timestamp": 0.0,\n        }\n    else:\n        raise RuntimeError(f"Unsupported business story media type: {media_path.name}")\n\n    data: dict[str, Any] = {\n        "business_connection_id": connection_id,\n        "content": json.dumps(content, ensure_ascii=False),\n        "active_period": str(int(config.get("period_seconds") or 24 * 60 * 60)),\n    }\n    caption = str(config.get("caption") or "").strip()\n    if caption:\n        data["caption"] = caption\n    files = {\n        attach_name: (\n            media_path.name,\n            media_path.open("rb"),\n            mimetypes.guess_type(str(media_path))[0] or "application/octet-stream",\n        )\n    }\n    try:\n        response = requests.post(\n            f"https://api.telegram.org/bot{bot_token}/postStory",\n            data=data,\n            files=files,\n            timeout=180,\n        )\n    finally:\n        files[attach_name][1].close()\n    try:\n        result = response.json()\n    except Exception:\n        result = {"ok": False, "description": response.text}\n    if response.status_code >= 400 or not bool(result.get("ok")):\n        description = str(result.get("description") or response.text or response.status_code)\n        raise RuntimeError(f"Bot API postStory failed: {description}")\n    return result\n\n\nasync def _story_targets_report(\n    client: TelegramClient,\n    *,\n    auth: dict[str, Any],\n    config: dict[str, Any],\n    log,\n    phase: str,\n    media_path: Path | None = None,\n    honor_delays: bool,\n) -> dict[str, Any]:\n    me = await client.get_me()\n    account = _account_info(me)\n    log(\n        f"Story {phase} account: {_account_label(account)} premium={account.get(\'premium\')}"\n    )\n    report = _build_story_report(\n        config,\n        phase=phase,\n        account=account,\n        media_path=media_path,\n    )\n    previous_story: dict[str, Any] | None = None\n    for idx, target_cfg in enumerate(config.get("targets") or []):\n        peer_ref = str(target_cfg.get("peer") or "").strip()\n        label = str(target_cfg.get("label") or peer_ref or f"target-{idx + 1}")\n        delay_seconds = max(0, int(target_cfg.get("delay_seconds") or 0))\n        publish_mode = str(target_cfg.get("mode") or "upload").strip().lower()\n        transport = str(target_cfg.get("transport") or "telethon").strip().lower()\n        blocking = _story_target_is_blocking(target_cfg, index=idx)\n        required = _story_target_is_required(target_cfg, blocking=blocking)\n        if publish_mode not in {"upload", "repost_previous"}:\n            publish_mode = "upload"\n        if honor_delays and delay_seconds:\n            log(f"⏳ Story target {label}: sleeping {delay_seconds}s before publish")\n            import asyncio\n\n            await asyncio.sleep(delay_seconds)\n        target_report = {\n            "peer": peer_ref,\n            "label": label,\n            "delay_seconds": delay_seconds,\n            "mode": publish_mode,\n            "transport": transport,\n            "blocking": blocking,\n            "required": required,\n            "period_seconds": int(config.get("period_seconds") or 24 * 60 * 60),\n            "pinned": bool(config.get("pinned")),\n            "ok": False,\n        }\n        try:\n            if transport == "telegram_business":\n                connection = _business_connection_for_target(auth, target_cfg)\n                if not connection:\n                    raise RuntimeError("business connection secret is missing")\n                if not bool(connection.get("is_enabled")):\n                    raise RuntimeError("business connection is disabled")\n                if not bool(connection.get("can_manage_stories")):\n                    raise RuntimeError("business connection lacks can_manage_stories")\n                target_report["connection_hash"] = str(\n                    target_cfg.get("business_connection_hash") or ""\n                ).strip()\n                if media_path is not None:\n                    result = _post_business_story(\n                        auth=auth,\n                        target_cfg=target_cfg,\n                        media_path=media_path,\n                        config=config,\n                    )\n                    target_report["story_id"] = _business_story_result_id(result)\n                    target_report["result"] = result.get("result")\n                    log(\n                        f"✅ Business story published to {label}"\n                        + (\n                            f" (story_id={target_report[\'story_id\']})"\n                            if target_report.get("story_id") is not None\n                            else ""\n                        )\n                    )\n                else:\n                    log(f"✅ Business story preflight passed for {label}")\n                target_report["ok"] = True\n                report["targets"].append(target_report)\n                continue\n            peer = await client.get_input_entity(peer_ref)\n            can_send = await client(functions.stories.CanSendStoryRequest(peer=peer))\n            target_report["can_send"] = (\n                can_send.to_dict() if hasattr(can_send, "to_dict") else str(can_send)\n            )\n            if media_path is not None:\n                send_kwargs: dict[str, Any]\n                if publish_mode == "repost_previous":\n                    if not previous_story:\n                        raise RuntimeError(\n                            "repost_previous target requires a successful prior story target"\n                        )\n                    source_peer = previous_story["peer"]\n                    source_story_id = int(previous_story["story_id"])\n                    media = types.InputMediaStory(peer=source_peer, id=source_story_id)\n                    target_report["source_story_id"] = source_story_id\n                    target_report["source_peer"] = previous_story.get("peer_ref")\n                    target_report["source_label"] = previous_story.get("label")\n                    send_kwargs = {\n                        "peer": peer,\n                        "media": media,\n                        "privacy_rules": [types.InputPrivacyValueAllowAll()],\n                        "pinned": bool(config.get("pinned")),\n                        "period": int(config.get("period_seconds") or 24 * 60 * 60),\n                        "fwd_from_id": source_peer,\n                        "fwd_from_story": source_story_id,\n                    }\n                else:\n                    media = await _input_media_for_path(client, media_path)\n                    send_kwargs = {\n                        "peer": peer,\n                        "media": media,\n                        "privacy_rules": [types.InputPrivacyValueAllowAll()],\n                        "pinned": bool(config.get("pinned")),\n                        "caption": str(config.get("caption") or "").strip() or None,\n                        "period": int(config.get("period_seconds") or 24 * 60 * 60),\n                    }\n                result = await client(\n                    functions.stories.SendStoryRequest(**send_kwargs)\n                )\n                target_report["story_id"] = _extract_story_id(result)\n                target_report["result"] = (\n                    result.to_dict() if hasattr(result, "to_dict") else str(result)\n                )\n                log(\n                    f"✅ Story published to {label}"\n                    + (\n                        f" (story_id={target_report[\'story_id\']})"\n                        if target_report.get("story_id") is not None\n                        else ""\n                    )\n                )\n                if target_report.get("story_id") is not None:\n                    previous_story = {\n                        "peer": peer,\n                        "peer_ref": peer_ref,\n                        "label": label,\n                        "story_id": int(target_report["story_id"]),\n                    }\n            else:\n                log(f"✅ Story preflight passed for {label}")\n            target_report["ok"] = True\n        except Exception as exc:\n            target_report["error"] = _format_story_error(exc, account=account)\n            log(\n                f"❌ Story {phase} failed for {label}: {target_report[\'error\']}"\n            )\n        report["targets"].append(target_report)\n    targets = report["targets"]\n    blocking_targets = [item for item in targets if item.get("blocking")]\n    required_targets = [\n        item for item in targets if item.get("blocking") or item.get("required")\n    ]\n    report["fanout_ok"] = bool(targets) and all(bool(item.get("ok")) for item in targets)\n    report["blocking_ok"] = bool(blocking_targets) and all(\n        bool(item.get("ok")) for item in blocking_targets\n    )\n    report["required_ok"] = bool(required_targets) and all(\n        bool(item.get("ok")) for item in required_targets\n    )\n    report["partial_ok"] = bool(report["blocking_ok"]) and not bool(report["fanout_ok"])\n    if phase == "preflight":\n        report["ok"] = bool(report["blocking_ok"])\n    else:\n        report["ok"] = bool(report["required_ok"])\n    if report["partial_ok"]:\n        failed_labels = [\n            str(item.get("label") or item.get("peer") or "?")\n            for item in targets\n            if not item.get("ok")\n        ]\n        phase_label = phase.capitalize()\n        failed_required_labels = [\n            str(item.get("label") or item.get("peer") or "?")\n            for item in targets\n            if not item.get("ok") and item.get("required")\n        ]\n        if failed_required_labels and phase == "preflight":\n            log(\n                f"⚠️ {phase_label} render gate passed; "\n                f"required fanout currently unavailable: {\', \'.join(failed_required_labels)}"\n            )\n        elif failed_required_labels:\n            log(\n                f"❌ {phase_label} required fanout failed: "\n                f"{\', \'.join(failed_required_labels)}"\n            )\n        else:\n            log(\n                f"⚠️ {phase_label} primary target passed; "\n                f"continuing despite best-effort fanout failures: {\', \'.join(failed_labels)}"\n            )\n    return report\n\n\nasync def preflight_story_publish_from_kaggle(\n    *,\n    search_roots: list[Path],\n    output_dir: Path,\n    log,\n) -> dict[str, Any] | None:\n    runtime = load_story_publish_runtime(search_roots=search_roots, log=log)\n    if not runtime:\n        return None\n\n    config = runtime["config"]\n    auth = runtime["auth"]\n    client = await _create_client(auth)\n    try:\n        report = await _story_targets_report(\n            client,\n            auth=auth,\n            config=config,\n            log=log,\n            phase="preflight",\n            media_path=None,\n            honor_delays=False,\n        )\n        if not report.get("fanout_ok", report.get("ok")):\n            write_story_publish_report(report, output_dir=output_dir)\n        return report\n    finally:\n        try:\n            await client.disconnect()\n        except Exception:\n            logger.warning("story_publish: failed to disconnect client", exc_info=True)\n\n\nasync def publish_story_from_kaggle(\n    *,\n    final_video_path: Path | None,\n    intro_path: Path | None,\n    posters: list[Path] | None,\n    search_roots: list[Path],\n    output_dir: Path,\n    log,\n) -> dict[str, Any] | None:\n    runtime = load_story_publish_runtime(search_roots=search_roots, log=log)\n    if not runtime:\n        return None\n\n    config = runtime["config"]\n    auth = runtime["auth"]\n    media_path = _story_media_path(\n        config,\n        final_video_path=final_video_path,\n        intro_path=intro_path,\n        posters=posters,\n    )\n    if str(config.get("mode") or "video").strip().lower() == "video":\n        upload_profile = _story_upload_profile(config)\n        if upload_profile == STORY_UPLOAD_PROFILE_NATIVE_HEVC:\n            media_path = _validate_native_story_video(media_path, log=log)\n        else:\n            media_path = _ensure_story_safe_video(\n                media_path,\n                output_dir=output_dir,\n                log=log,\n            )\n    client = await _create_client(auth)\n    try:\n        report = await _story_targets_report(\n            client,\n            auth=auth,\n            config=config,\n            log=log,\n            phase="publish",\n            media_path=media_path,\n            honor_delays=True,\n        )\n        write_story_publish_report(report, output_dir=output_dir)\n        return report\n    finally:\n        try:\n            await client.disconnect()\n        except Exception:\n            logger.warning("story_publish: failed to disconnect client", exc_info=True)'
        target.write_text(code, encoding='utf-8')
        print(f'✅ Wrote story_publish.py to {target}')
    except Exception as e:
        print(f'⚠️ Failed to write story_publish.py: {e}')

_ensure_story_publish_module()

def _ensure_story_gesture_overlay_module():
    target = Path('/kaggle/working/story_gesture_overlay.py')
    if target.exists():
        return
    try:
        code = 'from __future__ import annotations\n\nfrom dataclasses import dataclass\nimport math\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image, ImageChops, ImageDraw, ImageFilter, ImageFont\n\n\n@dataclass(frozen=True)\nclass GestureStep:\n    headline: str\n    subline: str\n    show_touch_cue: bool = False\n\n\nGESTURE_STEPS: tuple[GestureStep, ...] = (\n    GestureStep("Нажми и держи", "чтобы читать", show_touch_cue=True),\n    GestureStep("Держишь палец", "афиша на паузе"),\n    GestureStep("Веди по экрану", "чтобы промотать", show_touch_cue=True),\n)\nGESTURE_STEP_COUNT = len(GESTURE_STEPS)\n\n_FONT_PATHS: dict[str, Path | None] = {}\n_FONT_LOGGED = False\n_BACKGROUND_COLOR = (20, 20, 25)\n\n\ndef _font_candidates(search_roots: list[Path] | None, names: tuple[str, ...]) -> list[Path]:\n    candidates: list[Path] = []\n    seen: set[Path] = set()\n    roots = list(search_roots or [])\n    roots.extend(\n        [\n            Path.cwd(),\n            Path("/kaggle/working"),\n            Path("/kaggle/working/assets"),\n            Path("/kaggle/working/ro_znanie_fonts"),\n            Path("/kaggle/working/assets/ro_znanie_fonts"),\n        ]\n    )\n\n    for root in roots:\n        for base in (root, root / "assets", root / "ro_znanie_fonts", root / "assets" / "ro_znanie_fonts"):\n            if base in seen:\n                continue\n            seen.add(base)\n            for name in names:\n                candidate = base / name\n                if candidate.exists():\n                    candidates.append(candidate)\n\n    inp = Path("/kaggle/input")\n    if inp.exists():\n        for name in names:\n            for pattern in (\n                f"*/{name}",\n                f"*/assets/{name}",\n                f"*/ro_znanie_fonts/{name}",\n                f"*/assets/ro_znanie_fonts/{name}",\n            ):\n                candidates.extend(inp.glob(pattern))\n\n    candidates.extend(\n        [\n            Path("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"),\n            Path("/usr/share/fonts/truetype/dejavu/DejaVuSansCondensed.ttf"),\n        ]\n    )\n    return [path for path in candidates if path.exists()]\n\n\ndef _font_supports_cyrillic(font: ImageFont.FreeTypeFont) -> bool:\n    for ch in ("Я", "Ж", "Ю", "П"):\n        try:\n            bbox = font.getbbox(ch)\n        except Exception:\n            return False\n        if not bbox or (bbox[2] - bbox[0]) <= 0:\n            return False\n    return True\n\n\ndef _pick_font_path(role: str, *, search_roots: list[Path] | None = None) -> Path | None:\n    names = {\n        "headline": (\n            "Cygre-Medium.ttf",\n            "Cygre-SemiBold.ttf",\n            "Cygre-Bold.ttf",\n            "Cygre-Regular.ttf",\n        ),\n        "subline": (\n            "Cygre-Regular.ttf",\n            "Cygre-Book.ttf",\n            "Cygre-Medium.ttf",\n        ),\n    }.get(role, ("Cygre-Regular.ttf",))\n\n    for candidate in _font_candidates(search_roots, names):\n        try:\n            font = ImageFont.truetype(str(candidate), 32)\n        except Exception:\n            continue\n        if _font_supports_cyrillic(font):\n            return candidate\n    return None\n\n\ndef _load_font(\n    role: str,\n    size: int,\n    *,\n    search_roots: list[Path] | None = None,\n) -> ImageFont.FreeTypeFont:\n    global _FONT_LOGGED\n\n    path = _FONT_PATHS.get(role)\n    if path is None and role not in _FONT_PATHS:\n        path = _pick_font_path(role, search_roots=search_roots)\n        _FONT_PATHS[role] = path\n\n    if path is not None:\n        try:\n            font = ImageFont.truetype(str(path), int(size))\n            if not _FONT_LOGGED:\n                _FONT_LOGGED = True\n                print(f"✅ Story gesture font: {path}")\n            return font\n        except Exception:\n            pass\n\n    for fallback in (\n        Path("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"),\n        Path("/usr/share/fonts/truetype/dejavu/DejaVuSansCondensed.ttf"),\n    ):\n        if fallback.exists():\n            return ImageFont.truetype(str(fallback), int(size))\n    return ImageFont.load_default()\n\n\ndef _ease_out_cubic(value: float) -> float:\n    t = max(0.0, min(1.0, float(value)))\n    return 1.0 - ((1.0 - t) ** 3)\n\n\ndef _scale_alpha(color: tuple[int, int, int, int], scale: float) -> tuple[int, int, int, int]:\n    alpha = max(0, min(255, int(round(color[3] * max(0.0, min(1.0, scale))))))\n    return color[0], color[1], color[2], alpha\n\n\ndef _background_frame(width: int, height: int) -> np.ndarray:\n    if width <= 0 or height <= 0:\n        return np.zeros((0, 0, 3), dtype=np.uint8)\n    gradient = np.linspace(0.8, 1.0, num=height, dtype=np.float32)[:, None, None]\n    base = np.array(_BACKGROUND_COLOR, dtype=np.float32)[None, None, :]\n    bg = np.clip(base * gradient, 0, 255).astype(np.uint8)\n    return np.repeat(bg, width, axis=1)\n\n\ndef _paper_mask(frame_rgb: np.ndarray) -> np.ndarray:\n    height, width = frame_rgb.shape[:2]\n    bg = _background_frame(width, height)\n    diff = np.abs(frame_rgb.astype(np.int16) - bg.astype(np.int16)).sum(axis=2).astype(np.uint16)\n    mask = Image.fromarray(np.where(diff > 28, 255, 0).astype(np.uint8)).convert("L")\n    mask = mask.filter(ImageFilter.MaxFilter(7))\n    mask = mask.filter(ImageFilter.MaxFilter(7))\n    mask = mask.filter(ImageFilter.GaussianBlur(1.4))\n    return np.asarray(mask, dtype=np.uint8)\n\n\ndef _draw_touch_cue(\n    canvas: Image.Image,\n    *,\n    x: int,\n    y: int,\n    alpha_scale: float,\n    pulse: float,\n) -> Image.Image:\n    outer_scale = 1.0 + (0.05 * pulse)\n    mid_scale = 1.0 + (0.03 * pulse)\n    cue_r_outer = int(max(32, canvas.size[0] * 0.06) * outer_scale)\n    cue_r_mid = int(max(22, canvas.size[0] * 0.04) * mid_scale)\n    cue_r_dot = max(10, int(canvas.size[0] * 0.018))\n    finger_box = max(120, int(canvas.size[0] * 0.16))\n    finger_shift = int(canvas.size[1] * 0.004 * pulse)\n\n    overlay = Image.new("RGBA", canvas.size, (0, 0, 0, 0))\n    draw = ImageDraw.Draw(overlay, "RGBA")\n    ring = _scale_alpha((242, 239, 231, 52), alpha_scale * (0.85 + (0.15 * pulse)))\n    fill = _scale_alpha((242, 239, 231, 110), alpha_scale * (0.90 + (0.10 * pulse)))\n    fingertip = _scale_alpha((242, 239, 231, 190), alpha_scale)\n\n    draw.ellipse((x - cue_r_outer, y - cue_r_outer, x + cue_r_outer, y + cue_r_outer), outline=ring, width=2)\n    draw.ellipse((x - cue_r_mid, y - cue_r_mid, x + cue_r_mid, y + cue_r_mid), outline=ring, width=2)\n    draw.ellipse((x - cue_r_dot, y - cue_r_dot, x + cue_r_dot, y + cue_r_dot), fill=fill)\n\n    finger = Image.new("RGBA", (finger_box, finger_box), (0, 0, 0, 0))\n    finger_draw = ImageDraw.Draw(finger)\n    left = int(finger_box * 0.39)\n    top = int(finger_box * 0.11)\n    right = int(finger_box * 0.61)\n    bottom = int(finger_box * 0.76)\n    finger_draw.rounded_rectangle(\n        (left, top, right, bottom),\n        radius=max(14, int(finger_box * 0.10)),\n        fill=fingertip,\n    )\n    finger = finger.rotate(-28, resample=Image.Resampling.BICUBIC, expand=True)\n    overlay.alpha_composite(\n        finger,\n        dest=(x - finger.width // 2 - max(8, int(canvas.size[0] * 0.015)), y - finger.height // 2 + finger_shift),\n    )\n    return Image.alpha_composite(canvas, overlay)\n\n\ndef _render_gesture_layer(\n    size: tuple[int, int],\n    *,\n    step: GestureStep,\n    frame_index: int,\n    total_frames: int,\n    search_roots: list[Path] | None = None,\n) -> Image.Image:\n    width, height = size\n    progress = 1.0 if total_frames <= 1 else frame_index / float(max(total_frames - 1, 1))\n    appear = _ease_out_cubic((frame_index + 1) / 6.0)\n    settle = 1.0\n    if progress > 0.82:\n        settle = 1.0 - (0.15 * ((progress - 0.82) / 0.18))\n    alpha_scale = max(0.0, min(1.0, appear * settle))\n\n    pulse_progress = min(1.0, progress / 0.45) if step.show_touch_cue else 0.0\n    pulse = math.sin(pulse_progress * math.pi) if step.show_touch_cue else 0.0\n\n    canvas = Image.new("RGBA", size, (0, 0, 0, 0))\n    if step.show_touch_cue:\n        canvas = _draw_touch_cue(\n            canvas,\n            x=int(width * 0.15),\n            y=int(height * 0.81),\n            alpha_scale=alpha_scale,\n            pulse=pulse,\n        )\n\n    overlay = Image.new("RGBA", size, (0, 0, 0, 0))\n    shadow = Image.new("RGBA", size, (0, 0, 0, 0))\n    od = ImageDraw.Draw(overlay)\n    sd = ImageDraw.Draw(shadow)\n\n    headline_size = max(46, int(width * 0.073))\n    subline_size = max(34, int(width * 0.050))\n    line_gap = max(56, int(height * 0.078))\n    accent_w = max(132, int(width * 0.16))\n    text_x = int(width * 0.24)\n    text_y = int(height * 0.74)\n\n    headline_font = _load_font("headline", headline_size, search_roots=search_roots)\n    subline_font = _load_font("subline", subline_size, search_roots=search_roots)\n\n    shadow_color = _scale_alpha((0, 0, 0, 180), alpha_scale)\n    text_color = _scale_alpha((242, 239, 231, 238), alpha_scale)\n    sub_color = _scale_alpha((242, 239, 231, 210), alpha_scale)\n    accent = _scale_alpha((241, 228, 75, 205), alpha_scale * 0.90)\n\n    sd.text((text_x, text_y), step.headline, font=headline_font, fill=shadow_color)\n    sd.text((text_x, text_y + line_gap), step.subline, font=subline_font, fill=shadow_color)\n    shadow_blurred = shadow.filter(ImageFilter.GaussianBlur(14))\n    shadow_offset = ImageChops.offset(shadow_blurred, 0, max(4, int(height * 0.005)))\n\n    od.text((text_x, text_y), step.headline, font=headline_font, fill=text_color)\n    od.text((text_x, text_y + line_gap), step.subline, font=subline_font, fill=sub_color)\n    od.rounded_rectangle(\n        (\n            text_x,\n            text_y + line_gap + subline_size + 18,\n            text_x + accent_w,\n            text_y + line_gap + subline_size + 24,\n        ),\n        radius=3,\n        fill=accent,\n    )\n\n    canvas = Image.alpha_composite(canvas, shadow_offset)\n    return Image.alpha_composite(canvas, overlay)\n\n\ndef apply_story_gesture_frame(\n    frame_rgb: np.ndarray,\n    *,\n    step_index: int,\n    frame_index: int,\n    total_frames: int,\n    search_roots: list[str | Path] | None = None,\n) -> np.ndarray:\n    if step_index < 0 or step_index >= GESTURE_STEP_COUNT:\n        return frame_rgb\n    if frame_rgb is None or frame_rgb.ndim != 3 or frame_rgb.shape[2] != 3:\n        return frame_rgb\n\n    roots: list[Path] = []\n    if search_roots:\n        for root in search_roots:\n            try:\n                roots.append(Path(root))\n            except Exception:\n                continue\n\n    step = GESTURE_STEPS[step_index]\n    original = Image.fromarray(frame_rgb.astype(np.uint8)).convert("RGBA")\n    gesture_layer = _render_gesture_layer(\n        (frame_rgb.shape[1], frame_rgb.shape[0]),\n        step=step,\n        frame_index=frame_index,\n        total_frames=total_frames,\n        search_roots=roots,\n    )\n    paper_mask = Image.fromarray(_paper_mask(frame_rgb)).convert("L")\n    paper_layer = Image.composite(\n        original,\n        Image.new("RGBA", original.size, (0, 0, 0, 0)),\n        paper_mask,\n    )\n    composed = Image.alpha_composite(original, gesture_layer)\n    composed = Image.alpha_composite(composed, paper_layer)\n    return np.asarray(composed.convert("RGB"))'
        target.write_text(code, encoding='utf-8')
        print(f'✅ Wrote story_gesture_overlay.py to {target}')
    except Exception as e:
        print(f'⚠️ Failed to write story_gesture_overlay.py: {e}')

_ensure_story_gesture_overlay_module()

# Import from working dir (preferred)
sys.path.insert(0, '/kaggle/working')
from poster_overlay import apply_poster_overlay
from story_publish import preflight_story_publish_from_kaggle, publish_story_from_kaggle
from story_gesture_overlay import GESTURE_STEP_COUNT, apply_story_gesture_frame

OVERLAY_FONT_ROOTS = []
try:
    OVERLAY_FONT_ROOTS.append(SOURCE_FOLDER)
except Exception:
    pass
OVERLAY_FONT_ROOTS.append(Path('/kaggle/working'))
OVERLAY_FONT_ROOTS.append(Path('/kaggle/working/assets'))


REQUEST_HEADERS = {
    'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36',
    'Accept': 'image/avif,image/webp,image/apng,image/*,*/*;q=0.8',
}
SESSION = requests.Session()
SESSION.headers.update(REQUEST_HEADERS)

TELEGRAM_READY = False
TELEGRAM_ERROR = None
try:
    from telethon import TelegramClient
    from telethon.sessions import StringSession
    from kaggle_secrets import UserSecretsClient
    TELEGRAM_READY = True
except Exception as e:
    TELEGRAM_ERROR = e


async def download_via_telegram(filenames_map):
    if not TELEGRAM_READY or not filenames_map:
        return
    log(f"\n--- \U0001F535 Telegram: cache lookup ({len(filenames_map)} files) ---")
    try:
        secrets = UserSecretsClient()
        api_id = int(secrets.get_secret("TELEGRAM_API_ID"))
        api_hash = secrets.get_secret("TELEGRAM_API_HASH")
        session_str = secrets.get_secret("TELEGRAM_SESSION")
        channel_id = None
        try:
            cid = secrets.get_secret("SOURCE_CHANNEL_ID")
            if cid:
                channel_id = int(cid)
        except Exception:
            pass
    except Exception as e:
        log(f"[SKIP] Telegram secrets missing: {e}")
        return

    try:
        client = TelegramClient(StringSession(session_str), api_id, api_hash)
        await client.connect()
        if not await client.is_user_authorized():
            log("[ERROR] Telegram session invalid, skipping cache.")
            await client.disconnect()
            return
        target = channel_id if channel_id else 'me'
        target_name = 'CHANNEL' if channel_id else 'SAVED MESSAGES'
        log(f"   > Connected. Searching in: {target_name}")
        for fname, local_path in filenames_map.items():
            if os.path.exists(local_path):
                continue
            log(f"   > Search: {fname} ...")
            found_msg = None
            try:
                async for message in client.iter_messages(target, search=fname, limit=100):
                    if message.media:
                        found_msg = message
                        break
            except Exception as e:
                log(f"     [ERR] {e}")
            if found_msg:
                log('     [FOUND] Downloading...')
                await found_msg.download_media(file=local_path)
                log('     [DONE]')
            else:
                log('     [NOT FOUND]')
        await client.disconnect()
    except Exception as e:
        log(f"[TELEGRAM ERROR] {e}")


def _run_async(coro):
    try:
        running_loop = asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)

    result = {}
    error = {}

    def _runner():
        try:
            result["value"] = asyncio.run(coro)
        except Exception as exc:
            error["exc"] = exc

    t = threading.Thread(target=_runner)
    t.start()
    t.join()
    if "exc" in error:
        raise error["exc"]
    return result.get("value")


def _prepare_telegram_cache(urls, posters_dir):
    if not TELEGRAM_READY or not urls:
        return {}
    filenames_map = {}
    url_map = {}
    for idx, url in enumerate(urls, start=1):
        if not isinstance(url, str):
            continue
        fname = url.split('/')[-1].split('?')[0]
        if not fname:
            continue
        local_path = _telegram_cache_path(posters_dir, url, idx)
        if local_path is None:
            continue
        filenames_map[fname] = str(local_path)
        url_map[url] = local_path
    if filenames_map:
        _run_async(download_via_telegram(filenames_map))
    cache = {}
    for url, path in url_map.items():
        if path.exists():
            cache[url] = path
    if cache:
        log(f"✅ Telegram cache hits: {len(cache)}")
    return cache


def _resolve_image_path(filename: str):
    p = SOURCE_FOLDER / filename
    if p.exists():
        return p
    p = Path(KAGGLE_INPUT_ROOT) / filename
    if p.exists():
        return p
    for base in [SOURCE_FOLDER, Path(KAGGLE_INPUT_ROOT)]:
        try:
            matches = list(base.rglob(filename))
            if matches:
                return matches[0]
        except Exception:
            pass
    return None


def _guess_extension(url: str, content_type: str | None) -> str:
    ext = Path(url.split("?", 1)[0]).suffix.lower()
    if ext in (".jpg", ".jpeg", ".png", ".webp"):
        return ext
    if content_type:
        ext_guess = mimetypes.guess_extension(content_type.split(";", 1)[0].strip())
        if ext_guess in (".jpg", ".jpeg", ".png", ".webp"):
            return ext_guess
    return ".jpg"


def _poster_label(value: str | Path | None) -> str:
    raw = str(value or "").strip()
    if not raw:
        return "unknown"
    raw = raw.split("?", 1)[0].rstrip("/")
    name = Path(raw).name or raw
    if len(name) <= 72:
        return name
    return name[:32] + "..." + name[-24:]


def _telegram_cache_path(posters_dir: Path, url: str, idx: int) -> Path | None:
    if not isinstance(url, str) or not url:
        return None
    ext = Path(url.split("?", 1)[0]).suffix.lower()
    if ext not in {".jpg", ".jpeg", ".png", ".webp"}:
        ext = ".jpg"
    digest = hashlib.md5(url.encode("utf-8")).hexdigest()[:16]
    return posters_dir / f"tg_{idx:02d}_{digest}{ext}"


def _render_readiness_label(ready_scenes: int, total_scenes: int) -> str:
    if total_scenes <= 0:
        return "NO_SCENES"
    if ready_scenes <= 0:
        return "FAIL"
    if ready_scenes == total_scenes:
        return "FULL"
    if ready_scenes * 2 > total_scenes:
        return "GO (degraded)"
    if ready_scenes * 2 == total_scenes:
        return "BORDERLINE"
    return "HIGH_RISK"


def _log_poster_preflight(source_reports, scene_reports):
    if source_reports:
        log("\n--- 🟩 Poster preflight: source availability ---")
        for report in source_reports:
            status = "✅" if report.get("ok") else "❌"
            label = report.get("label") or "unknown"
            source = report.get("source") or "unknown"
            detail = str(report.get("detail") or "").strip()
            tail = f" · {detail}" if detail else ""
            log(f"{status} {label} · {source}{tail}")
    if scene_reports:
        log("\n--- 🟩 Poster preflight: scene readiness ---")
        for report in scene_reports:
            status = "✅" if report.get("ok") else "❌"
            scene_no = int(report.get("scene", -1)) + 1
            detail = str(report.get("detail") or "poster missing").strip()
            log(f"{status} Scene {scene_no} · {detail}")
    ready_sources = sum(1 for report in source_reports if report.get("ok"))
    total_sources = len(source_reports)
    ready_scenes = sum(1 for report in scene_reports if report.get("ok"))
    total_scenes = len(scene_reports)
    missing = [str(int(report.get("scene", -1)) + 1) for report in scene_reports if not report.get("ok")]
    readiness = _render_readiness_label(ready_scenes, total_scenes)
    missing_tail = f"; missing scenes: {', '.join(missing)}" if missing else ""
    log(
        f"Poster preflight summary: sources {ready_sources}/{total_sources} ready; "
        f"scenes {ready_scenes}/{total_scenes} ready; render readiness: {readiness}{missing_tail}"
    )


def _poster_label(value: str | Path | None) -> str:
    raw = str(value or "").strip()
    if not raw:
        return "unknown"
    raw = raw.split("?", 1)[0].rstrip("/")
    name = Path(raw).name or raw
    if len(name) <= 72:
        return name
    return name[:32] + "..." + name[-24:]


def _telegram_cache_path(posters_dir: Path, url: str, idx: int) -> Path | None:
    if not isinstance(url, str) or not url:
        return None
    ext = Path(url.split("?", 1)[0]).suffix.lower()
    if ext not in {".jpg", ".jpeg", ".png", ".webp"}:
        ext = ".jpg"
    digest = hashlib.md5(url.encode("utf-8")).hexdigest()[:16]
    return posters_dir / f"tg_{idx:02d}_{digest}{ext}"


def _render_readiness_label(ready_scenes: int, total_scenes: int) -> str:
    if total_scenes <= 0:
        return "NO_SCENES"
    if ready_scenes <= 0:
        return "FAIL"
    if ready_scenes == total_scenes:
        return "FULL"
    if ready_scenes * 2 > total_scenes:
        return "GO (degraded)"
    if ready_scenes * 2 == total_scenes:
        return "BORDERLINE"
    return "HIGH_RISK"


def _log_poster_preflight(source_reports, scene_reports):
    if source_reports:
        log("\n--- 🟩 Poster preflight: source availability ---")
        for report in source_reports:
            status = "✅" if report.get("ok") else "❌"
            label = report.get("label") or "unknown"
            source = report.get("source") or "unknown"
            detail = str(report.get("detail") or "").strip()
            tail = f" · {detail}" if detail else ""
            log(f"{status} {label} · {source}{tail}")
    if scene_reports:
        log("\n--- 🟩 Poster preflight: scene readiness ---")
        for report in scene_reports:
            status = "✅" if report.get("ok") else "❌"
            scene_no = int(report.get("scene", -1)) + 1
            detail = str(report.get("detail") or "poster missing").strip()
            log(f"{status} Scene {scene_no} · {detail}")
    ready_sources = sum(1 for report in source_reports if report.get("ok"))
    total_sources = len(source_reports)
    ready_scenes = sum(1 for report in scene_reports if report.get("ok"))
    total_scenes = len(scene_reports)
    missing = [str(int(report.get("scene", -1)) + 1) for report in scene_reports if not report.get("ok")]
    readiness = _render_readiness_label(ready_scenes, total_scenes)
    missing_tail = f"; missing scenes: {', '.join(missing)}" if missing else ""
    log(
        f"Poster preflight summary: sources {ready_sources}/{total_sources} ready; "
        f"scenes {ready_scenes}/{total_scenes} ready; render readiness: {readiness}{missing_tail}"
    )


def _poster_label(value: str | Path | None) -> str:
    raw = str(value or "").strip()
    if not raw:
        return "unknown"
    raw = raw.split("?", 1)[0].rstrip("/")
    name = Path(raw).name or raw
    if len(name) <= 72:
        return name
    return name[:32] + "..." + name[-24:]


def _telegram_cache_path(posters_dir: Path, url: str, idx: int) -> Path | None:
    if not isinstance(url, str) or not url:
        return None
    ext = Path(url.split("?", 1)[0]).suffix.lower()
    if ext not in {".jpg", ".jpeg", ".png", ".webp"}:
        ext = ".jpg"
    digest = hashlib.md5(url.encode("utf-8")).hexdigest()[:16]
    return posters_dir / f"tg_{idx:02d}_{digest}{ext}"


def _render_readiness_label(ready_scenes: int, total_scenes: int) -> str:
    if total_scenes <= 0:
        return "NO_SCENES"
    if ready_scenes <= 0:
        return "FAIL"
    if ready_scenes == total_scenes:
        return "FULL"
    if ready_scenes * 2 > total_scenes:
        return "GO (degraded)"
    if ready_scenes * 2 == total_scenes:
        return "BORDERLINE"
    return "HIGH_RISK"


def _log_poster_preflight(source_reports, scene_reports):
    if source_reports:
        log("\n--- 🟩 Poster preflight: source availability ---")
        for report in source_reports:
            status = "✅" if report.get("ok") else "❌"
            label = report.get("label") or "unknown"
            source = report.get("source") or "unknown"
            detail = str(report.get("detail") or "").strip()
            tail = f" · {detail}" if detail else ""
            log(f"{status} {label} · {source}{tail}")
    if scene_reports:
        log("\n--- 🟩 Poster preflight: scene readiness ---")
        for report in scene_reports:
            status = "✅" if report.get("ok") else "❌"
            scene_no = int(report.get("scene", -1)) + 1
            detail = str(report.get("detail") or "poster missing").strip()
            log(f"{status} Scene {scene_no} · {detail}")
    ready_sources = sum(1 for report in source_reports if report.get("ok"))
    total_sources = len(source_reports)
    ready_scenes = sum(1 for report in scene_reports if report.get("ok"))
    total_scenes = len(scene_reports)
    missing = [str(int(report.get("scene", -1)) + 1) for report in scene_reports if not report.get("ok")]
    readiness = _render_readiness_label(ready_scenes, total_scenes)
    missing_tail = f"; missing scenes: {', '.join(missing)}" if missing else ""
    log(
        f"Poster preflight summary: sources {ready_sources}/{total_sources} ready; "
        f"scenes {ready_scenes}/{total_scenes} ready; render readiness: {readiness}{missing_tail}"
    )


def _poster_label(value: str | Path | None) -> str:
    raw = str(value or "").strip()
    if not raw:
        return "unknown"
    raw = raw.split("?", 1)[0].rstrip("/")
    name = Path(raw).name or raw
    if len(name) <= 72:
        return name
    return name[:32] + "..." + name[-24:]


def _telegram_cache_path(posters_dir: Path, url: str, idx: int) -> Path | None:
    if not isinstance(url, str) or not url:
        return None
    ext = Path(url.split("?", 1)[0]).suffix.lower()
    if ext not in {".jpg", ".jpeg", ".png", ".webp"}:
        ext = ".jpg"
    digest = hashlib.md5(url.encode("utf-8")).hexdigest()[:16]
    return posters_dir / f"tg_{idx:02d}_{digest}{ext}"


def _render_readiness_label(ready_scenes: int, total_scenes: int) -> str:
    if total_scenes <= 0:
        return "NO_SCENES"
    if ready_scenes <= 0:
        return "FAIL"
    if ready_scenes == total_scenes:
        return "FULL"
    if ready_scenes * 2 > total_scenes:
        return "GO (degraded)"
    if ready_scenes * 2 == total_scenes:
        return "BORDERLINE"
    return "HIGH_RISK"


def _log_poster_preflight(source_reports, scene_reports):
    if source_reports:
        log("\n--- 🟩 Poster preflight: source availability ---")
        for report in source_reports:
            status = "✅" if report.get("ok") else "❌"
            label = report.get("label") or "unknown"
            source = report.get("source") or "unknown"
            detail = str(report.get("detail") or "").strip()
            tail = f" · {detail}" if detail else ""
            log(f"{status} {label} · {source}{tail}")
    if scene_reports:
        log("\n--- 🟩 Poster preflight: scene readiness ---")
        for report in scene_reports:
            status = "✅" if report.get("ok") else "❌"
            scene_no = int(report.get("scene", -1)) + 1
            detail = str(report.get("detail") or "poster missing").strip()
            log(f"{status} Scene {scene_no} · {detail}")
    ready_sources = sum(1 for report in source_reports if report.get("ok"))
    total_sources = len(source_reports)
    ready_scenes = sum(1 for report in scene_reports if report.get("ok"))
    total_scenes = len(scene_reports)
    missing = [str(int(report.get("scene", -1)) + 1) for report in scene_reports if not report.get("ok")]
    readiness = _render_readiness_label(ready_scenes, total_scenes)
    missing_tail = f"; missing scenes: {', '.join(missing)}" if missing else ""
    log(
        f"Poster preflight summary: sources {ready_sources}/{total_sources} ready; "
        f"scenes {ready_scenes}/{total_scenes} ready; render readiness: {readiness}{missing_tail}"
    )


def _poster_label(value: str | Path | None) -> str:
    raw = str(value or "").strip()
    if not raw:
        return "unknown"
    raw = raw.split("?", 1)[0].rstrip("/")
    name = Path(raw).name or raw
    if len(name) <= 72:
        return name
    return name[:32] + "..." + name[-24:]


def _telegram_cache_path(posters_dir: Path, url: str, idx: int) -> Path | None:
    if not isinstance(url, str) or not url:
        return None
    ext = Path(url.split("?", 1)[0]).suffix.lower()
    if ext not in {".jpg", ".jpeg", ".png", ".webp"}:
        ext = ".jpg"
    digest = hashlib.md5(url.encode("utf-8")).hexdigest()[:16]
    return posters_dir / f"tg_{idx:02d}_{digest}{ext}"


def _render_readiness_label(ready_scenes: int, total_scenes: int) -> str:
    if total_scenes <= 0:
        return "NO_SCENES"
    if ready_scenes <= 0:
        return "FAIL"
    if ready_scenes == total_scenes:
        return "FULL"
    if ready_scenes * 2 > total_scenes:
        return "GO (degraded)"
    if ready_scenes * 2 == total_scenes:
        return "BORDERLINE"
    return "HIGH_RISK"


def _log_poster_preflight(source_reports, scene_reports):
    if source_reports:
        log("\n--- 🟩 Poster preflight: source availability ---")
        for report in source_reports:
            status = "✅" if report.get("ok") else "❌"
            label = report.get("label") or "unknown"
            source = report.get("source") or "unknown"
            detail = str(report.get("detail") or "").strip()
            tail = f" · {detail}" if detail else ""
            log(f"{status} {label} · {source}{tail}")
    if scene_reports:
        log("\n--- 🟩 Poster preflight: scene readiness ---")
        for report in scene_reports:
            status = "✅" if report.get("ok") else "❌"
            scene_no = int(report.get("scene", -1)) + 1
            detail = str(report.get("detail") or "poster missing").strip()
            log(f"{status} Scene {scene_no} · {detail}")
    ready_sources = sum(1 for report in source_reports if report.get("ok"))
    total_sources = len(source_reports)
    ready_scenes = sum(1 for report in scene_reports if report.get("ok"))
    total_scenes = len(scene_reports)
    missing = [str(int(report.get("scene", -1)) + 1) for report in scene_reports if not report.get("ok")]
    readiness = _render_readiness_label(ready_scenes, total_scenes)
    missing_tail = f"; missing scenes: {', '.join(missing)}" if missing else ""
    log(
        f"Poster preflight summary: sources {ready_sources}/{total_sources} ready; "
        f"scenes {ready_scenes}/{total_scenes} ready; render readiness: {readiness}{missing_tail}"
    )


def _poster_label(value: str | Path | None) -> str:
    raw = str(value or "").strip()
    if not raw:
        return "unknown"
    raw = raw.split("?", 1)[0].rstrip("/")
    name = Path(raw).name or raw
    if len(name) <= 72:
        return name
    return name[:32] + "..." + name[-24:]


def _telegram_cache_path(posters_dir: Path, url: str, idx: int) -> Path | None:
    if not isinstance(url, str) or not url:
        return None
    ext = Path(url.split("?", 1)[0]).suffix.lower()
    if ext not in {".jpg", ".jpeg", ".png", ".webp"}:
        ext = ".jpg"
    digest = hashlib.md5(url.encode("utf-8")).hexdigest()[:16]
    return posters_dir / f"tg_{idx:02d}_{digest}{ext}"


def _render_readiness_label(ready_scenes: int, total_scenes: int) -> str:
    if total_scenes <= 0:
        return "NO_SCENES"
    if ready_scenes <= 0:
        return "FAIL"
    if ready_scenes == total_scenes:
        return "FULL"
    if ready_scenes * 2 > total_scenes:
        return "GO (degraded)"
    if ready_scenes * 2 == total_scenes:
        return "BORDERLINE"
    return "HIGH_RISK"


def _log_poster_preflight(source_reports, scene_reports):
    if source_reports:
        log("\n--- 🟩 Poster preflight: source availability ---")
        for report in source_reports:
            status = "✅" if report.get("ok") else "❌"
            label = report.get("label") or "unknown"
            source = report.get("source") or "unknown"
            detail = str(report.get("detail") or "").strip()
            tail = f" · {detail}" if detail else ""
            log(f"{status} {label} · {source}{tail}")
    if scene_reports:
        log("\n--- 🟩 Poster preflight: scene readiness ---")
        for report in scene_reports:
            status = "✅" if report.get("ok") else "❌"
            scene_no = int(report.get("scene", -1)) + 1
            detail = str(report.get("detail") or "poster missing").strip()
            log(f"{status} Scene {scene_no} · {detail}")
    ready_sources = sum(1 for report in source_reports if report.get("ok"))
    total_sources = len(source_reports)
    ready_scenes = sum(1 for report in scene_reports if report.get("ok"))
    total_scenes = len(scene_reports)
    missing = [str(int(report.get("scene", -1)) + 1) for report in scene_reports if not report.get("ok")]
    readiness = _render_readiness_label(ready_scenes, total_scenes)
    missing_tail = f"; missing scenes: {', '.join(missing)}" if missing else ""
    log(
        f"Poster preflight summary: sources {ready_sources}/{total_sources} ready; "
        f"scenes {ready_scenes}/{total_scenes} ready; render readiness: {readiness}{missing_tail}"
    )


def _poster_label(value: str | Path | None) -> str:
    raw = str(value or "").strip()
    if not raw:
        return "unknown"
    raw = raw.split("?", 1)[0].rstrip("/")
    name = Path(raw).name or raw
    if len(name) <= 72:
        return name
    return name[:32] + "..." + name[-24:]


def _telegram_cache_path(posters_dir: Path, url: str, idx: int) -> Path | None:
    if not isinstance(url, str) or not url:
        return None
    ext = Path(url.split("?", 1)[0]).suffix.lower()
    if ext not in {".jpg", ".jpeg", ".png", ".webp"}:
        ext = ".jpg"
    digest = hashlib.md5(url.encode("utf-8")).hexdigest()[:16]
    return posters_dir / f"tg_{idx:02d}_{digest}{ext}"


def _render_readiness_label(ready_scenes: int, total_scenes: int) -> str:
    if total_scenes <= 0:
        return "NO_SCENES"
    if ready_scenes <= 0:
        return "FAIL"
    if ready_scenes == total_scenes:
        return "FULL"
    if ready_scenes * 2 > total_scenes:
        return "GO (degraded)"
    if ready_scenes * 2 == total_scenes:
        return "BORDERLINE"
    return "HIGH_RISK"


def _log_poster_preflight(source_reports, scene_reports):
    if source_reports:
        log("\n--- 🟩 Poster preflight: source availability ---")
        for report in source_reports:
            status = "✅" if report.get("ok") else "❌"
            label = report.get("label") or "unknown"
            source = report.get("source") or "unknown"
            detail = str(report.get("detail") or "").strip()
            tail = f" · {detail}" if detail else ""
            log(f"{status} {label} · {source}{tail}")
    if scene_reports:
        log("\n--- 🟩 Poster preflight: scene readiness ---")
        for report in scene_reports:
            status = "✅" if report.get("ok") else "❌"
            scene_no = int(report.get("scene", -1)) + 1
            detail = str(report.get("detail") or "poster missing").strip()
            log(f"{status} Scene {scene_no} · {detail}")
    ready_sources = sum(1 for report in source_reports if report.get("ok"))
    total_sources = len(source_reports)
    ready_scenes = sum(1 for report in scene_reports if report.get("ok"))
    total_scenes = len(scene_reports)
    missing = [str(int(report.get("scene", -1)) + 1) for report in scene_reports if not report.get("ok")]
    readiness = _render_readiness_label(ready_scenes, total_scenes)
    missing_tail = f"; missing scenes: {', '.join(missing)}" if missing else ""
    log(
        f"Poster preflight summary: sources {ready_sources}/{total_sources} ready; "
        f"scenes {ready_scenes}/{total_scenes} ready; render readiness: {readiness}{missing_tail}"
    )


def main_pipeline():
    """Full pipeline."""

    import os
    import shutil

    # Kaggle output download via API can fail when outputs contain hundreds of MB of PNG frames.
    # Default behavior: cleanup bulky intermediates unless explicitly kept for debugging.
    keep_debug = bool(os.getenv("CRUMPLE_KEEP_DEBUG"))
    
    payload_path = SOURCE_FOLDER / "payload.json"
    
    # Fallback for missing payload.json
    if not payload_path.exists():
        log(f"⚠️ payload.json not found at {payload_path}. Generatng fallback.")
        payload_path = WORKING_DIR / "payload.json"
        fallback_scenes = []
        # Find images in source folder
        for ext in ["*.jpg", "*.png"]:
            for img in SOURCE_FOLDER.glob(ext):
                fallback_scenes.append({"image": img.name, "text": f"Fallback {img.stem}"})
        # Sort and take top 4
        fallback_scenes.sort(key=lambda x: x["image"])
        fallback_data = {"scenes": fallback_scenes[:4]}
        with open(payload_path, "w") as f:
            json.dump(fallback_data, f)
        log(f"✅ Generated fallback payload at {payload_path} with {len(fallback_scenes)} scenes")
    
    try:
        payload = json.loads(payload_path.read_text())
    except Exception as e:
        log(f"❌ Failed to read payload: {e}")
        return False

    scenes = payload.get("scenes", [])
    log(f"Payload scenes: {len(scenes)}")
    for idx, scene in enumerate(scenes[:5]):
        imgs = scene.get("images")
        if isinstance(imgs, str):
            imgs = [imgs]
        log(f"Scene {idx}: images={len(imgs or [])} image={scene.get('image')}")
    
    intro = payload.get("intro") or {}
    cover = payload.get("cover") or {}
    default_pattern = getattr(pattern_preview, "PATTERN_STICKER", "STICKER")
    intro_text = intro.get("text") or cover.get("title") or "АФИША"
    intro_pattern = intro.get("pattern") or cover.get("pattern") or default_pattern
    intro_cities = intro.get("cities") or cover.get("cities") or []
    intro_path = WORKING_DIR / "intro.png"

    selection_params = payload.get("selection_params") or {}
    is_test_mode = False
    if isinstance(selection_params, dict):
        is_test_mode = bool(selection_params.get("test") or selection_params.get("is_test"))
        mode = selection_params.get("mode")
        if isinstance(mode, str) and mode.lower() == "test":
            is_test_mode = True
    if not is_test_mode:
        source_name = str(SOURCE_FOLDER.name).lower()
        if "test" in source_name:
            is_test_mode = True

    if is_test_mode:
        test_limit = None
        if isinstance(intro, dict):
            test_limit = intro.get("count")
        if isinstance(selection_params, dict):
            test_limit = selection_params.get("test_scenes") or selection_params.get("test_limit") or test_limit
        if not isinstance(test_limit, int) or test_limit <= 0:
            test_limit = 3
        scenes = scenes[:test_limit]
        log(f"Test mode scene limit: {test_limit}")
    log(f"Scenes after test limit: {len(scenes)}")

    posters = [intro_path]

    posters_dir = WORKING_DIR / "posters"
    posters_dir.mkdir(parents=True, exist_ok=True)
    download_cache: dict[str, Path] = {}
    download_status: dict[str, dict] = {}
    poster_source_reports = []
    scene_reports = []
    telegram_cache = {}
    url_map: dict[str, Path] = {}
    urls_to_cache = []
    seen_urls = set()
    for scene in scenes[:12]:
        imgs = scene.get('images')
        if isinstance(imgs, str):
            imgs = [imgs]
        for u in (imgs or []):
            if isinstance(u, str) and u.startswith('http') and u not in seen_urls:
                seen_urls.add(u)
                urls_to_cache.append(u)
    if urls_to_cache:
        filenames_map = {}
        for idx, url in enumerate(urls_to_cache, start=1):
            fname = url.split('/')[-1].split('?')[0]
            if not fname:
                continue
            local_path = _telegram_cache_path(posters_dir, url, idx)
            if local_path is None:
                continue
            filenames_map[fname] = str(local_path)
            url_map[url] = local_path
        if TELEGRAM_READY and filenames_map:
            _run_async(download_via_telegram(filenames_map))
        elif not TELEGRAM_READY:
            log(f"[SKIP] Telegram cache disabled: {TELEGRAM_ERROR}")
        telegram_cache = {url: path for url, path in url_map.items() if path.exists()}
        if telegram_cache:
            log(f"✅ Telegram cache hits: {len(telegram_cache)}")


    def _download_poster(url: str, idx: int):
        if not url:
            return None
        if url in download_cache:
            cached_path = download_cache[url]
            if url not in download_status:
                download_status[url] = {
                    "ok": True,
                    "source": "download-cache",
                    "detail": f"reused {cached_path.name}",
                }
            return cached_path
        cached = telegram_cache.get(url)
        if cached is not None and cached.exists():
            download_cache[url] = cached
            download_status[url] = {
                "ok": True,
                "source": "telegram-cache",
                "detail": f"cached {cached.name}",
            }
            return cached
        last_err = None
        for attempt in range(3):
            try:
                resp = SESSION.get(url, timeout=(5, 30), allow_redirects=True)
                resp.raise_for_status()
                content_type = resp.headers.get("content-type") or ""
                data = resp.content
                if not data:
                    raise RuntimeError("empty response")
                if content_type and not content_type.startswith("image/"):
                    raise RuntimeError(f"unexpected content-type: {content_type}")
                if len(data) < 2048:
                    raise RuntimeError(f"response too small: {len(data)} bytes")
                ext = _guess_extension(url, content_type)
                out_path = url_map.get(url)
                if out_path is None:
                    digest = hashlib.md5(url.encode("utf-8")).hexdigest()[:8]
                    out_path = posters_dir / f"poster_{idx}_{digest}{ext}"
                elif out_path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".webp"}:
                    out_path = out_path.with_suffix(ext)
                    url_map[url] = out_path
                out_path.write_bytes(data)
                download_cache[url] = out_path
                download_status[url] = {
                    "ok": True,
                    "source": "http",
                    "detail": f"downloaded {out_path.name}",
                }
                return out_path
            except Exception as e:
                last_err = e
                log(
                    f"⚠️ Failed to download poster (attempt {attempt + 1}/3): {url} ({e})"
                )
                time.sleep(1.5 * (attempt + 1))
        if last_err is not None:
            download_status[url] = {
                "ok": False,
                "source": "http",
                "detail": f"{type(last_err).__name__}: {last_err}",
            }
            log(f"⚠️ Poster download failed after retries: {url} ({last_err})")
        return None

    for idx, url in enumerate(urls_to_cache, start=1):
        resolved = _download_poster(url, idx)
        status = download_status.get(url) or {}
        poster_source_reports.append(
            {
                "ok": bool(resolved),
                "label": _poster_label(url),
                "source": str(status.get("source") or ("missing" if not resolved else "http")),
                "detail": str(status.get("detail") or ("ready" if resolved else "not downloaded")),
            }
        )

    # Collect scene posters
    for i, scene in enumerate(scenes[:12]):  # Max 12 as per user spec
        img_name = scene.get('image')
        images = scene.get("images")
        if isinstance(images, str):
            images = [images]
        found = None
        selected_label = None
        selected_source = None
        if img_name:
            found = _resolve_image_path(img_name)
            if found:
                selected_label = _poster_label(img_name)
                selected_source = "local-image"

        if not found and images:
            for candidate in images:
                if not candidate:
                    continue
                if isinstance(candidate, str) and candidate.startswith("http"):
                    found = _download_poster(candidate, i + 1)
                    status = download_status.get(candidate) or {}
                    if found:
                        selected_label = _poster_label(candidate)
                        selected_source = str(status.get("source") or "http")
                elif isinstance(candidate, str):
                    found = _resolve_image_path(candidate)
                    if found:
                        selected_label = _poster_label(candidate)
                        selected_source = "local-image"
                if found:
                    break

        if not found:
            # Fallback to posterN naming
            for ext in [".jpg", ".png", ".jpeg"]:
                local = SOURCE_FOLDER / f"poster{i+1}{ext}"
                if local.exists():
                    found = local
                    selected_label = local.name
                    selected_source = "fallback-local"
                    break

        overlay = scene.get("poster_overlay")
        overlay_text = None
        highlight_title = None
        if isinstance(overlay, dict):
            overlay_text = overlay.get("text")
            missing = overlay.get("missing")
            if isinstance(missing, list):
                highlight_title = "title" in {str(part).strip().casefold() for part in missing if isinstance(part, str)}
        if found and isinstance(overlay_text, str) and overlay_text.strip():
            try:
                found = apply_poster_overlay(
                    found,
                    text=overlay_text,
                    out_dir=posters_dir,
                    search_roots=OVERLAY_FONT_ROOTS,
                    highlight_title=highlight_title,
                )
            except Exception as e:
                log(f"⚠️ Overlay failed for scene {i}: {e}")

        if found:
            posters.append(found)
            scene_reports.append(
                {
                    "scene": i,
                    "ok": True,
                    "detail": f"{selected_label or Path(found).name} via {selected_source or 'local'}",
                }
            )
        else:
            scene_reports.append(
                {
                    "scene": i,
                    "ok": False,
                    "detail": "poster missing",
                }
            )
            log(f"⚠️ Poster not found for scene {i}")

    _log_poster_preflight(poster_source_reports, scene_reports)

    _log_poster_preflight(poster_source_reports, scene_reports)

    _log_poster_preflight(poster_source_reports, scene_reports)

    _log_poster_preflight(poster_source_reports, scene_reports)

    _log_poster_preflight(poster_source_reports, scene_reports)

    _log_poster_preflight(poster_source_reports, scene_reports)

    _log_poster_preflight(poster_source_reports, scene_reports)

    # Check for Final scene
    final_img = _resolve_image_path("Final.png")
    if final_img:
        posters.append(final_img)
        log(f"✅ Appended final scene: {final_img}")
    else:
        log("⚠️ Final.png not found; skipping final scene")
        
    log(f"Posters to render: {len(posters)}")

    render_samples = BLENDER_SAMPLES
    render_pct = 100
    frame_start = FRAME_START
    frame_end = FRAME_END
    extra_args = None
    if is_test_mode:
        test_limit = None
        if isinstance(intro, dict):
            test_limit = intro.get("count")
        if isinstance(selection_params, dict):
            test_limit = selection_params.get("test_scenes") or selection_params.get("test_limit") or test_limit
        if not isinstance(test_limit, int) or test_limit <= 0:
            test_limit = 3
        scenes = scenes[:test_limit]
        log(f"Test mode scene limit: {test_limit}")
        render_samples = 15
        render_pct = 84
        frame_end = min(FRAME_END, 40)
        extra_args = {"substeps": 4, "iters": 6}
        log(
            f"⚡ Test mode render: samples={render_samples}, resolution_pct={render_pct}, frame_end={frame_end}"
        )

    if pattern_preview and is_test_mode and intro_pattern == default_pattern:
        intro_pattern = f"{default_pattern}_YELLOW"

    # Generate Intro via pattern_preview or CSS layout
    generate_intro_image(intro_pattern, intro_text, intro_cities, intro_path, intro=intro, cover=cover)

    story_smoke_only = False
    if isinstance(selection_params, dict):
        story_smoke_only = bool(
            selection_params.get("story_smoke_only")
            or selection_params.get("story_publish_smoke_only")
        )
    if story_smoke_only:
        log("🧪 Story smoke mode: publishing image from Kaggle without video render")
        try:
            story_report = _run_async(
                publish_story_from_kaggle(
                    final_video_path=None,
                    intro_path=intro_path,
                    posters=posters,
                    search_roots=[SOURCE_FOLDER, Path(KAGGLE_INPUT_ROOT)],
                    output_dir=WORKING_DIR,
                    log=log,
                )
            )
        except Exception as e:
            story_report = {
                "ok": False,
                "mode": "image",
                "error": f"{type(e).__name__}: {e}",
                "targets": [],
            }
            (WORKING_DIR / "story_publish_report.json").write_text(
                json.dumps(story_report, ensure_ascii=False, indent=2),
                encoding="utf-8",
            )
            log(f"❌ Story smoke publish crashed: {story_report['error']}")
        if not story_report:
            log("FATAL: story smoke mode requested but story publish config is missing")
            return False
        log(f"Story publish status: {'OK' if story_report.get('ok') else 'FAIL'}")
        return bool(story_report.get("ok"))

    story_preflight = _run_async(
        preflight_story_publish_from_kaggle(
            search_roots=[SOURCE_FOLDER, Path(KAGGLE_INPUT_ROOT)],
            output_dir=WORKING_DIR,
            log=log,
        )
    )
    if story_preflight:
        log(f"Story preflight status: {'OK' if story_preflight.get('ok') else 'FAIL'}")
        if not story_preflight.get("ok"):
            log("FATAL: story publish preflight failed before video render")
            return False

    # Build segments
    segments = []
    
    # Define is_last for the first segment (Intro)
    is_last = (len(posters) <= 1)
    
    # 1. Intro (Cover) - Starts Flat, Crumples UP (Fold)
    # We want Intro to just be visible then crumple away?
    # Or start crumpled and uncrumple?
    # If stem='frames_0', run_blender_render produced 1..64 (1=Crumpled, 64=Flat).
    # Standard flow: Unfold (Ball->Flat) -> Show -> Fold (Flat->Ball).
    # Cover usually: Starts Flat (no unfold), Shows, then Folds (to transition to next).
    segments.append(SegmentPlan(
        stem="frames_0",
        unfold_len=0,     # Start Flat
        fold_len=24,      # Crumple away
        hold_flat=48 if is_last else 2,
        hold_ball=2,
        easing_unfold="linear",
        easing_fold="ease_out_quad"
    ))
    
    # 2. Main Scenes + Outro
    # Iterate posters starting from index 1
    for i in range(1, len(posters)):
        is_last = (i == len(posters) - 1)
        
        # If this is the LAST poster (Outro/Final), we want it to Stay Open.
        # Unfold -> Hold Flat (Long) -> No Fold
        
        segments.append(SegmentPlan(
            stem=f"frames_{i}",
            unfold_len=0 if is_last else 28,
            fold_len=0 if is_last else 28,      # No fold for Outro
            hold_flat=48 if is_last else 4,     # Hold slightly longer for posters
            hold_ball=0 if is_last else 2,
            easing_unfold="ease_in_out_back",
            easing_fold="ease_in_out_back"
        ))
    
    # Render Blender frames
    for i, poster in enumerate(posters):
        frames_dir = WORKING_DIR / f"frames_{i}"
        expected_frames = frame_end - frame_start + 1
        # Check if already rendered (re-entrancy)
        if frames_dir.exists() and len(list(frames_dir.glob("*.png"))) >= expected_frames:
            log(f"Skipping render for {poster.name} (cached)")
            continue
            
        success = run_blender_render(
            poster,
            frames_dir,
            render_samples,
            render_pct,
            frame_start=frame_start,
            frame_end=frame_end,
            extra_args=extra_args,
        )
        if not success:
            log(f"Skipping poster {i}")
    
    # Assemble sequence
    log("Assembling showreel...")
    sequence = []
    gesture_step_index = 0
    pending_gesture_step = None
    pending_gesture_total = 0
    pending_gesture_offset = 0
    gesture_fold_lead_frames = 8
    
    for seg_index, seg in enumerate(segments):
        frames_dir = WORKING_DIR / seg.stem
        frames = load_segment_frames(frames_dir, frame_start, frame_end)
        
        if not frames:
            log(f"No frames for {seg.stem}")
            continue
        
        ball = frames[0]
        flat = frames[-1]
        
        # Unfold (Ball -> Flat)
        if seg.unfold_len > 0:
            unfold = render_motion(frames, seg.unfold_len, seg.easing_unfold, False, True)
            if pending_gesture_step is not None and pending_gesture_total > 0:
                unfold = [
                    apply_story_gesture_frame(
                        frame,
                        step_index=pending_gesture_step,
                        frame_index=pending_gesture_offset + idx,
                        total_frames=pending_gesture_total,
                        search_roots=OVERLAY_FONT_ROOTS,
                    )
                    for idx, frame in enumerate(unfold)
                ]
                pending_gesture_step = None
                pending_gesture_total = 0
                pending_gesture_offset = 0
            sequence.extend(unfold)
        
        # Hold Flat
        for _ in range(seg.hold_flat):
            sequence.append(flat)
        
        # Fold (Flat -> Ball)
        if seg.fold_len > 0:
            fold = render_motion(frames, seg.fold_len, seg.easing_fold, True, True)
            sequence.extend(fold)
        
        next_unfold_len = 0
        if seg_index + 1 < len(segments):
            next_unfold_len = int(max(0, segments[seg_index + 1].unfold_len))

        if gesture_step_index < GESTURE_STEP_COUNT and next_unfold_len > 0 and seg.fold_len > 0 and sequence:
            fold_lead = min(gesture_fold_lead_frames, seg.fold_len)
            if fold_lead > 0:
                interstitial_total = fold_lead + seg.hold_ball + next_unfold_len
                fold_start = len(sequence) - fold_lead
                sequence[fold_start:] = [
                    apply_story_gesture_frame(
                        frame,
                        step_index=gesture_step_index,
                        frame_index=idx,
                        total_frames=interstitial_total,
                        search_roots=OVERLAY_FONT_ROOTS,
                    )
                    for idx, frame in enumerate(sequence[fold_start:])
                ]
                pending_gesture_step = gesture_step_index
                pending_gesture_total = interstitial_total
                pending_gesture_offset = fold_lead + seg.hold_ball
                active_gesture_step = gesture_step_index
                active_interstitial_total = interstitial_total
                gesture_step_index += 1
            else:
                active_gesture_step = None
                active_interstitial_total = 0
        else:
            active_gesture_step = None
            active_interstitial_total = 0

        # Hold Ball
        ball_frames = [ball for _ in range(seg.hold_ball)]
        next_unfold_len = 0
        if seg_index + 1 < len(segments):
            next_unfold_len = int(max(0, segments[seg_index + 1].unfold_len))
        if active_gesture_step is not None and ball_frames:
            ball_frames = [
                apply_story_gesture_frame(
                    frame,
                    step_index=active_gesture_step,
                    frame_index=(pending_gesture_offset - len(ball_frames)) + idx,
                    total_frames=active_interstitial_total,
                    search_roots=OVERLAY_FONT_ROOTS,
                )
                for idx, frame in enumerate(ball_frames)
            ]
        sequence.extend(ball_frames)

        if not keep_debug:
            try:
                shutil.rmtree(frames_dir)
            except Exception:
                pass
    
    log(f"Total frames: {len(sequence)}")
    
    if not sequence:
        log("FATAL: No frames!")
        return False
    
    seq_dir = WORKING_DIR / "sequence"
    if seq_dir.exists():
        import shutil
        shutil.rmtree(seq_dir)
    seq_dir.mkdir(exist_ok=True)
    
    for i, frame in enumerate(sequence):
        frame_bgr = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
        cv2.imwrite(str(seq_dir / f"frame_{i+1:06d}.png"), frame_bgr)
    
    video_path = WORKING_DIR / "crumple_video.mp4"
    video_ok = render_video_from_frames(seq_dir, video_path, fps=24)
    if not video_ok:
        log("FATAL: Video render failed")
        return False
    if not video_path.exists() or video_path.stat().st_size == 0:
        log(f"FATAL: Video file missing or empty: {video_path}")
        return False

    if not keep_debug:
        try:
            shutil.rmtree(seq_dir)
        except Exception:
            pass
    
    final_path = WORKING_DIR / "crumple_video_final.mp4"
    audio_files = sorted(SOURCE_FOLDER.glob("*.mp3"))
    if is_test_mode:
        test_limit = None
        if isinstance(intro, dict):
            test_limit = intro.get("count")
        if isinstance(selection_params, dict):
            test_limit = selection_params.get("test_scenes") or selection_params.get("test_limit") or test_limit
        if not isinstance(test_limit, int) or test_limit <= 0:
            test_limit = 3
        scenes = scenes[:test_limit]
        log(f"Test mode scene limit: {test_limit}")

    preferred_audio = SOURCE_FOLDER / AUDIO_FILENAME
    if not preferred_audio.exists():
        available_audio = [path.name for path in audio_files]
        log(f"FATAL: Required audio missing: {preferred_audio}")
        log(f"Available mp3 files: {available_audio}")
        return False

    log(f"🎵 Using audio: {preferred_audio.name} @ {AUDIO_START_SEC}s")
    audio_ok = add_audio(video_path, preferred_audio, final_path, start_sec=AUDIO_START_SEC)
    if not audio_ok:
        log("FATAL: Audio merge failed")
        return False
    
    if final_path.exists():
        size_kb = final_path.stat().st_size / 1024
        log(f"Final video: {size_kb:.1f} KB")
        if size_kb < 500:
            log("FAIL: Video too small!")
            return False
        try:
            story_report = _run_async(
                publish_story_from_kaggle(
                    final_video_path=final_path,
                    intro_path=intro_path,
                    posters=posters,
                    search_roots=[SOURCE_FOLDER, Path(KAGGLE_INPUT_ROOT)],
                    output_dir=WORKING_DIR,
                    log=log,
                )
            )
        except Exception as e:
            story_report = {
                "ok": False,
                "mode": "video",
                "error": f"{type(e).__name__}: {e}",
                "targets": [],
            }
            (WORKING_DIR / "story_publish_report.json").write_text(
                json.dumps(story_report, ensure_ascii=False, indent=2),
                encoding="utf-8",
            )
            log(f"❌ Story publish crashed: {story_report['error']}")
        if story_report is not None:
            log(f"Story publish status: {'OK' if story_report.get('ok') else 'FAIL'}")
            if not story_report.get("ok"):
                return False
        if not keep_debug:
            try:
                shutil.rmtree(posters_dir)
            except Exception:
                pass
        return True
    
    return False

log("=" * 50)
log("Starting CrumpleVideo Pipeline")
log("=" * 50)

try:
    success = main_pipeline()
except Exception as e:
    import traceback
    traceback.print_exc()
    success = False

if success:
    log("Pipeline completed!")
else:
    log("Pipeline failed!")
